# AInstein component: LLM QA

In [1]:
import sys
from pathlib import Path

path_project = Path.cwd().parent
sys.path.append(str(path_project))

In [2]:
from AInstein import (
    get_llm_azure_openai, # conexión al modelo GPT
    get_settings, # conexión a las configuraciones de los modelos
    BigQueryManager
)

In [3]:
import re
import json
from collections import defaultdict

import ipywidgets as widgets
import pandas as pd
from IPython.display import display, Markdown, HTML
from openpyxl import load_workbook
from io import BytesIO
import copy

In [7]:
!pip install google-cloud-bigquery
!pip install pandas

# Settings

In [5]:
# Settings
ENVIRONMENT: str = 'bdb-gcp-sbx-ia'
WORKPLACE_PROJECT_ID: str = 'geo-cargas_laborales'

# Obtener las configuraciones del proyecto
settings = get_settings(WORKPLACE_PROJECT_ID, environment=ENVIRONMENT)

# Models

In [6]:
# Crear instancias de los modelos
llm = get_llm_azure_openai(settings)  # Modelo de Azure OpenAI

# Responses

## General

In [20]:
import re
import json
import copy
from io import BytesIO
from openpyxl import load_workbook

# ===========================================================================
#  ProcesadorTranscripcionTeams
#  v6 — Transcripción Teams con resumen + completado de campos
# ===========================================================================

class ProcesadorTranscripcionTeams:
    """
    Procesa archivos VTT de Microsoft Teams para levantar actividades,
    enriquecerlas con IA y generar resúmenes confirmables.

    CAMBIOS v6
    ----------
    1. La evaluación de completitud de actividades ahora considera tanto
       el campo "nombre" como el campo "descripcion" de forma conjunta,
       dando un análisis más preciso sobre si se necesita más detalle.
    2. Al cargar la transcripción de Teams se muestra un resumen tabular
       de las actividades detectadas (nombre, duración). Luego se le
       pregunta al usuario si desea completar los campos faltantes
       (frecuencia, volumen, autonomía, proceso del área, observaciones)
       para cada actividad extraída. Las actividades completadas se
       agregan al Excel final junto con las registradas manualmente.
    """

    # -----------------------------------------------------------------------
    # CONSTANTES COMPARTIDAS
    # -----------------------------------------------------------------------
    FRECUENCIAS_VALIDAS = [
        "Diario", "Semanal", "Quincenal",
        "Mensual", "Bimensual", "Trimestral", "Semestral", "Anual"
    ]

    # -----------------------------------------------------------------------
    # 1. LECTURA Y LIMPIEZA DEL VTT
    # -----------------------------------------------------------------------
    def leer_archivo_vtt(self, contenido):
        conversaciones = []
        lineas = contenido.splitlines()

        timestamp_actual = None
        buffer_texto = []

        for linea in lineas:
            linea = linea.strip()

            if (
                not linea
                or linea == "WEBVTT"
                or re.match(r"^[a-f0-9\-]+\/\d+\-\d+$", linea)
            ):
                continue

            if "-->" in linea:
                if timestamp_actual and buffer_texto:
                    conversaciones.append({
                        "timestamp": timestamp_actual,
                        "texto": " ".join(buffer_texto).strip()
                    })
                    buffer_texto = []

                inicio = linea.split("-->")[0].strip()
                timestamp_actual = inicio.split(".")[0]
                continue

            linea = re.sub(r"<v[^>]*>", "", linea)
            linea = re.sub(r"</v>", "", linea)
            linea = re.sub(r"^[A-Za-zÁÉÍÓÚÑáéíóúñ\s,]+:\s*", "", linea)

            if linea:
                buffer_texto.append(linea)

        if timestamp_actual and buffer_texto:
            conversaciones.append({
                "timestamp": timestamp_actual,
                "texto": " ".join(buffer_texto).strip()
            })

        return conversaciones

    # -----------------------------------------------------------------------
    # 2. EXTRACCIÓN DE ACTIVIDADES (SIN IA)
    # -----------------------------------------------------------------------
    def extraer_actividades(self, conversaciones):
        actividades = []
        actividad_actual = None

        patrones_inicio = [
            r"\binicio\b", r"\biniciar\b", r"\bcomienzo\b",
            r"\bempiezo\b", r"\bvoy a iniciar\b"
        ]
        patrones_fin = [
            r"\bfinalizo\b", r"\btermino\b",
            r"\bterminar\b", r"\bfinalizar\b"
        ]

        for conv in conversaciones:
            texto = conv["texto"].lower()
            timestamp = conv["timestamp"]

            if any(re.search(p, texto) for p in patrones_inicio):
                actividad_actual = {
                    "descripcion": conv["texto"],
                    "inicio": timestamp,
                    "fin": None
                }
            elif actividad_actual and any(re.search(p, texto) for p in patrones_fin):
                actividad_actual["fin"] = timestamp
                actividades.append(actividad_actual)
                actividad_actual = None

        return actividades

    # -----------------------------------------------------------------------
    # 3. CÁLCULO DE DURACIÓN
    # -----------------------------------------------------------------------
    def calcular_duracion_minutos(self, inicio, fin):
        h1, m1, s1 = map(int, inicio.split(":"))
        h2, m2, s2 = map(int, fin.split(":"))
        t1 = h1 * 3600 + m1 * 60 + s1
        t2 = h2 * 3600 + m2 * 60 + s2
        return max((t2 - t1) // 60, 0)

    # -----------------------------------------------------------------------
    # 4. PIPELINE DE PROCESAMIENTO
    # -----------------------------------------------------------------------
    def procesar_archivo(self, contenido):
        conversaciones = self.leer_archivo_vtt(contenido)
        actividades = self.extraer_actividades(conversaciones)

        resultado = []
        for a in actividades:
            if not a["fin"]:
                continue
            duracion = self.calcular_duracion_minutos(a["inicio"], a["fin"])
            if duracion <= 0:
                continue
            resultado.append({
                "actividad": a["descripcion"],
                "inicio": a["inicio"],
                "fin": a["fin"],
                "duracion_min": duracion
            })
        return resultado

    # -----------------------------------------------------------------------
    # 5. ENRIQUECIMIENTO CON IA (UNIDAD + PHVA)
    # -----------------------------------------------------------------------
    def _parse_json_seguro(self, texto):
        texto = texto.strip()
        if not texto:
            raise ValueError("❌ El LLM devolvió una respuesta vacía")
        if texto.startswith("```"):
            texto = re.sub(r"```json|```", "", texto).strip()
        inicio = min(
            [i for i in [texto.find("["), texto.find("{")] if i != -1],
            default=-1
        )
        if inicio > 0:
            texto = texto[inicio:]
        try:
            return json.loads(texto)
        except json.JSONDecodeError:
            print("❌ JSON inválido devuelto por el LLM")
            print("Respuesta cruda:")
            print(texto)
            raise

    def enriquecer_actividades(self, actividades, cargo, vicepresidencia):
        prompt = f"""
Eres un analista experto en Levantamiento de Cargas Laborales.

Contexto del colaborador:
- Cargo: {cargo}
- Vicepresidencia: {vicepresidencia}

Para cada actividad, debes:
- unidad_medida (ej: solicitudes, informes, reuniones, casos, desarrollos)
- phva (Planear, Hacer, Verificar, Actuar)

⚠️ REGLAS ESTRICTAS:
- Devuelve EXCLUSIVAMENTE un JSON válido
- NO incluyas texto antes o después
- NO expliques nada
- NO uses markdown
- Devuelve una LISTA del mismo tamaño que la entrada

Formato exacto de salida:
[
  {{
    "nombre": "texto corto",
    "unidad_medida": "texto",
    "phva": "Planear | Hacer | Verificar | Actuar"
  }}
]

Actividades de entrada:
{json.dumps(actividades, indent=2, ensure_ascii=False)}
"""
        reply = llm.invoke(prompt)
        return self._parse_json_seguro(reply.content)

    # -----------------------------------------------------------------------
    # 5b. DETECCIÓN DE ACTIVIDADES IRRELEVANTES O QUE NECESITEN MÁS DETALLE
    # -----------------------------------------------------------------------
    def _evaluar_relevancia_y_detalle(self, actividad, cargo, vicepresidencia):
        """
        Evalúa si la actividad es potencialmente irrelevante para el
        levantamiento o si el NOMBRE no es suficientemente representativo
        dado el conjunto completo de variables operativas que la componen.

        Ya no existe campo "descripcion" — la evaluación se basa en:
          • El nombre de la actividad.
          • Las variables operativas: frecuencia, volumen, duracion_min,
            autonomia, proceso_area, observaciones.
        El LLM considera todo el contexto para determinar si el nombre
        refleja adecuadamente lo que la actividad representa.

        Devuelve un dict con:
            - es_irrelevante      : bool
            - necesita_detalle    : bool
            - razon_irrelevante   : str
            - razon_detalle       : str
            - nombres_sugeridos   : list[str]  — 2-3 nombres alternativos
                                     cuando necesita_detalle es True
        """
        nombre = actividad.get("nombre", "").strip()

        # Variables operativas que aportan contexto para evaluar el nombre
        vars_operativas = {
            k: v for k, v in actividad.items()
            if k not in ("nombre", "descripcion", "_origen",
                         "_consolidada", "_ocurrencias",
                         "_duraciones_originales", "_analisis_interno")
            and v not in (None, "", [])
        }

        prompt = f"""
Eres un experto en Levantamiento de Cargas Laborales.

Analiza la siguiente actividad de un colaborador con:
- Cargo: {cargo}
- Vicepresidencia: {vicepresidencia}

Nombre de la actividad: "{nombre}"

Variables operativas registradas:
{json.dumps(vars_operativas, indent=2, ensure_ascii=False)}

Tu tarea es evaluar DOS cosas:

1. ¿Es esta actividad IRRELEVANTE para un levantamiento de cargas?
   Considera irrelevantes: actividades personales (ir al baño, tomar agua,
   almorzar), actividades fuera del ámbito laboral, actividades demasiado
   triviales que no representan carga laboral real.

2. ¿El NOMBRE es suficientemente representativo considerando TODO el
   contexto (frecuencia, volumen, duración, proceso del área, autonomía,
   observaciones y el cargo del colaborador)?
   - Si el nombre es genérico pero las variables operativas aclaran bien
     de qué se trata, NO marques como que necesita detalle.
   - Solo marca "necesita_detalle: true" si el nombre sigue siendo
     ambiguo o podría confundirse con otra actividad, incluso teniendo
     en cuenta el resto de variables.
   - Cuando necesita_detalle sea true, propón 2 o 3 nombres alternativos
     concretos y descriptivos que reflejen mejor lo que las variables
     sugieren. Los nombres deben ser cortos (máx. 8 palabras), claros y
     en el idioma español.

Responde EXCLUSIVAMENTE con un JSON válido:

{{
  "es_irrelevante": true | false,
  "necesita_detalle": true | false,
  "razon_irrelevante": "explicación breve o cadena vacía",
  "razon_detalle": "explicación breve de por qué el nombre no es representativo, o cadena vacía",
  "nombres_sugeridos": ["Nombre sugerido 1", "Nombre sugerido 2", "Nombre sugerido 3"]
}}

Si necesita_detalle es false, "nombres_sugeridos" debe ser [].
"""
        reply = llm.invoke(prompt)
        try:
            resultado = self._parse_json_seguro(reply.content)
            # Normalizar campo por si el LLM devuelve "sugerencia" en vez de lista
            if "nombres_sugeridos" not in resultado:
                resultado["nombres_sugeridos"] = []
            return resultado
        except Exception:
            return {
                "es_irrelevante": False,
                "necesita_detalle": False,
                "razon_irrelevante": "",
                "razon_detalle": "",
                "nombres_sugeridos": [],
            }

    def _manejar_alerta_relevancia(self, actividad, evaluacion):
        """
        Muestra las alertas de relevancia/detalle al usuario y le permite
        tomar una decisión.

        Caso IRRELEVANTE:
          Opciones: descartar | conservar.

        Caso NECESITA_DETALLE:
          Muestra los nombres alternativos sugeridos por el LLM como
          opciones numeradas. El usuario elige uno, o conserva el nombre
          actual, o descarta la actividad.

        Retorna:
            ("continuar", actividad) — sigue con la actividad (nombre
                                       actualizado o no)
            ("descartar", None)      — el usuario quiere descartar
        """
        hay_alerta = evaluacion["es_irrelevante"] or evaluacion["necesita_detalle"]
        if not hay_alerta:
            return "continuar", actividad

        print("\n" + "─" * 60)

        # ── Caso 1: actividad irrelevante ──────────────────────────────────
        if evaluacion["es_irrelevante"]:
            print("⚠️  ACTIVIDAD POSIBLEMENTE NO RELEVANTE")
            print("─" * 60)
            print(f"   {evaluacion['razon_irrelevante']}")
            print("─" * 60)
            print("\n¿Qué deseas hacer?")
            print("  1) Conservarla tal como está")
            print("  2) Descartarla")
            while True:
                opcion = input("\nSelecciona una opción (1 / 2): ").strip()
                if opcion == "1":
                    print("✅ Se conserva la actividad.")
                    return "continuar", actividad
                elif opcion == "2":
                    print("🗑️  Actividad descartada.")
                    return "descartar", None
                else:
                    print("⚠️ Opción no válida. Por favor selecciona 1 o 2.")

        # ── Caso 2: nombre poco representativo ────────────────────────────
        if evaluacion["necesita_detalle"]:
            print("📝  NOMBRE POCO REPRESENTATIVO")
            print("─" * 60)
            print(f"   {evaluacion['razon_detalle']}")
            print("─" * 60)

            sugeridos = [s for s in evaluacion.get("nombres_sugeridos", []) if s]

            if sugeridos:
                print("\n💡 Nombres alternativos sugeridos:")
                for i, s in enumerate(sugeridos, start=1):
                    print(f"   {i}) {s}")

                n_opciones = len(sugeridos)
                print(f"   {n_opciones + 1}) Conservar el nombre actual")
                print(f"   {n_opciones + 2}) Descartar la actividad")

                while True:
                    opcion = input(
                        f"\nSelecciona una opción (1 - {n_opciones + 2}): "
                    ).strip()
                    if opcion.isdigit():
                        idx = int(opcion)
                        if 1 <= idx <= n_opciones:
                            nuevo_nombre = sugeridos[idx - 1]
                            actividad["nombre"] = nuevo_nombre
                            print(f"✅ Nombre actualizado a: \"{nuevo_nombre}\"")
                            return "continuar", actividad
                        elif idx == n_opciones + 1:
                            print("✅ Se conserva el nombre actual.")
                            return "continuar", actividad
                        elif idx == n_opciones + 2:
                            print("🗑️  Actividad descartada.")
                            return "descartar", None
                    print(f"⚠️ Opción no válida. Elige entre 1 y {n_opciones + 2}.")
            else:
                # Sin sugerencias (LLM no generó ninguna)
                print("\n¿Qué deseas hacer?")
                print("  1) Conservar el nombre actual")
                print("  2) Descartar la actividad")
                while True:
                    opcion = input("\nSelecciona una opción (1 / 2): ").strip()
                    if opcion == "1":
                        print("✅ Se conserva el nombre actual.")
                        return "continuar", actividad
                    elif opcion == "2":
                        print("🗑️  Actividad descartada.")
                        return "descartar", None
                    else:
                        print("⚠️ Opción no válida. Por favor selecciona 1 o 2.")

        return "continuar", actividad

    # -----------------------------------------------------------------------
    # 6. CONSTRUCCIÓN DE RESUMEN ESTRUCTURADO
    # -----------------------------------------------------------------------
    def construir_resumen_actividades(self, actividades_enriquecidas):
        resumen = ""
        for i, act in enumerate(actividades_enriquecidas, start=1):
            resumen += f"""
Actividad {i}:
- Descripción: {act['nombre']}
- Frecuencia: {act.get('frecuencia', 'No especificada')}
- Duración (minutos): {act.get('duracion_min')}
- Proceso del área: {act.get('proceso_area')}
- Volumen: {act.get('volumen', 'N/A')}
- Unidad de Medida: {act.get('unidad_medida')}
- Tipo de actividad (PHVA): {act.get('phva')}
- Autonomía: {act.get('autonomia', 'N/A')}%
"""
            if act.get("observaciones"):
                resumen += f"- Observaciones: {act['observaciones']}\n"
        return resumen

    # -----------------------------------------------------------------------
    # 7. RESUMEN PARA CONFIRMACIÓN DEL USUARIO
    # -----------------------------------------------------------------------
    def resumen_para_confirmacion(self, actividades_enriquecidas, contexto):
        resumen_actividades = self.construir_resumen_actividades(
            actividades_enriquecidas
        )
        prompt = f"""
Eres un asistente de Levantamiento de Cargas Laborales.

Contexto del colaborador:
- Cargo: {contexto['cargo']}
- Vicepresidencia: {contexto['vicepresidencia']}

A continuación se presenta un resumen de las actividades
identificadas a partir de reuniones y la información ingresada
por el colaborador.

{resumen_actividades}

Instrucciones:
- Resume la información de forma clara y ordenada
- No combines el Volumen con la Unidad de Medida,
  dalos por separado ya que el volumen es respecto a la Frecuencia
- Usa lenguaje sencillo y no técnico
- No agregues información nueva
- No hagas cálculos adicionales
- No corrijas ortografía a menos que sean la misma palabra en distinta capitalización
- Finaliza preguntando si la información es correcta o si desea hacer ajustes
"""
        reply = llm.invoke(prompt)
        return reply.content

    # -----------------------------------------------------------------------
    # 8. LECTURA DE ARCHIVO VTT DESDE FILEUPLOAD (JUPYTER)
    # -----------------------------------------------------------------------
    def obtener_contenido_vtt(self, upload_widget):
        if not upload_widget.value:
            raise ValueError("No se ha subido ningún archivo")
        valor = upload_widget.value
        archivo = valor[0] if isinstance(valor, tuple) else list(valor.values())[0]
        contenido_raw = archivo["content"]
        if isinstance(contenido_raw, memoryview):
            return contenido_raw.tobytes().decode("utf-8")
        elif isinstance(contenido_raw, bytes):
            return contenido_raw.decode("utf-8")
        else:
            raise TypeError("Tipo de contenido no soportado")

    # -----------------------------------------------------------------------
    # 8b. FUSIÓN DE ACTIVIDADES DIARIAS
    # -----------------------------------------------------------------------
    def fusionar_inputs_usuario(
        self, actividades_base, actividades_enriquecidas, inputs_usuario
    ):
        actividades_finales = []
        for base, ia, user in zip(
            actividades_base, actividades_enriquecidas, inputs_usuario
        ):
            actividades_finales.append({
                "nombre": ia["nombre"],
                "duracion_min": base["duracion_min"],
                "unidad_medida": ia["unidad_medida"],
                "phva": ia["phva"],
                "frecuencia": user["frecuencia"],
                "volumen": user["volumen"],
                "proceso_area": user["proceso_area"],
                "autonomia": user["autonomia"],
                "observaciones": user.get("observaciones", "")
            })
        return actividades_finales

    # -----------------------------------------------------------------------
    # 9. CONFIRMAR LA INFORMACIÓN POR PARTE DEL USUARIO
    # -----------------------------------------------------------------------
    def confirmar_informacion(self, resumen_texto):
        print("📋 RESUMEN PARA CONFIRMACIÓN\n")
        print(resumen_texto)
        respuesta = input(
            "\n¿La información es correcta? (si / no): "
        ).strip().lower()
        return respuesta == "si"

    # -----------------------------------------------------------------------
    # 10. CALCULAR MÉTRICAS
    # -----------------------------------------------------------------------
    def calcular_metricas(self, actividades):
        DIAS_LABORALES_MES = 21
        MINUTOS_JORNADA_MES = 220 * 60
        JORNADA_LABORAL_DIARIA = 8.5

        FACTOR_FRECUENCIA = {
            "Diario": 1,
            "Semanal": 1 / 5,
            "Quincenal": 1 / 10,
            "Mensual": 1 / 21,
            "Bimensual": 1 / 42,
            "Trimestral": 1 / 63,
            "Semestral": 1 / 126,
            "Anual": 1 / 252
        }

        carga_w_por_phva = {}
        cantidad_por_phva = {}
        carga_w_por_frecuencia = {}
        cantidad_por_frecuencia = {}
        carga_w_por_proceso_area = {}
        cantidad_por_proceso_area = {}

        total_carga_w_sin_tm = 0
        total_carga_trabajo_individual = 0
        total_minutos_diarios = 0

        for act in actividades:
            tiempo_base = act["duracion_min"]
            volumen = act["volumen"]
            frecuencia = act["frecuencia"]
            autonomia = act["autonomia"] / 100
            phva = act.get("phva", "SIN CLASIFICAR")
            proceso_area = act.get("proceso_area", "SIN CLASIFICAR")

            factor = FACTOR_FRECUENCIA.get(frecuencia, 0)
            minutos_diarios = tiempo_base * volumen * factor
            minutos_mes = minutos_diarios * DIAS_LABORALES_MES
            carga_w_sin_tm = minutos_mes / MINUTOS_JORNADA_MES
            carga_trabajo_individual = carga_w_sin_tm * autonomia

            act["metricas"] = {
                "minutos_diarios": round(minutos_diarios, 2),
                "minutos_mes": round(minutos_mes, 2),
                "carga_w_sin_tm": round(carga_w_sin_tm, 4),
                "carga_trabajo_individual": round(carga_trabajo_individual, 4)
            }

            total_carga_w_sin_tm += carga_w_sin_tm
            total_carga_trabajo_individual += carga_trabajo_individual
            total_minutos_diarios += minutos_diarios

            carga_w_por_phva[phva] = carga_w_por_phva.get(phva, 0) + carga_w_sin_tm
            cantidad_por_phva[phva] = cantidad_por_phva.get(phva, 0) + 1
            carga_w_por_frecuencia[frecuencia] = (
                carga_w_por_frecuencia.get(frecuencia, 0) + carga_w_sin_tm
            )
            cantidad_por_frecuencia[frecuencia] = (
                cantidad_por_frecuencia.get(frecuencia, 0) + 1
            )
            carga_w_por_proceso_area[proceso_area] = (
                carga_w_por_proceso_area.get(proceso_area, 0) + carga_w_sin_tm
            )
            cantidad_por_proceso_area[proceso_area] = (
                cantidad_por_proceso_area.get(proceso_area, 0) + 1
            )

        almuerzo = 1 / 8
        baño_agua_etc = (3 * 5) / 60
        break_15min = 15 / 60
        latencia_software = 10 / 60
        cliente_ext_o_int = 20 / 60
        tiempo_muerto = sum([
            almuerzo / JORNADA_LABORAL_DIARIA,
            baño_agua_etc / JORNADA_LABORAL_DIARIA,
            break_15min / JORNADA_LABORAL_DIARIA,
            latencia_software / JORNADA_LABORAL_DIARIA,
            cliente_ext_o_int / JORNADA_LABORAL_DIARIA
        ])
        factor_tiempo_neto = round(1 - tiempo_muerto, 4)

        def porcentajes(dic):
            total = sum(dic.values())
            return {
                k: round(v / total, 4) if total > 0 else 0
                for k, v in dic.items()
            }

        horas_diarias_requeridas = total_minutos_diarios / 60
        horas_netas_por_persona = JORNADA_LABORAL_DIARIA * factor_tiempo_neto

        return {
            "carga_trabajo_phva": {k: round(v, 4) for k, v in carga_w_por_phva.items()},
            "cantidad_actividades_phva": cantidad_por_phva,
            "porcentaje_actividades_phva": porcentajes(carga_w_por_phva),
            "carga_trabajo_frecuencia": {k: round(v, 4) for k, v in carga_w_por_frecuencia.items()},
            "cantidad_actividades_frecuencia": cantidad_por_frecuencia,
            "porcentaje_actividades_frecuencia": porcentajes(carga_w_por_frecuencia),
            "carga_trabajo_proceso_area": {
                k: round(v, 4) for k, v in carga_w_por_proceso_area.items()
            },
            "cantidad_actividades_proceso_area": cantidad_por_proceso_area,
            "porcentaje_actividades_proceso_area": porcentajes(carga_w_por_proceso_area),
            "total_carga_w_sin_tm": round(total_carga_w_sin_tm, 4),
            "total_carga_trabajo_individual": round(total_carga_trabajo_individual, 4),
            "minutos_diarios_empleados": round(total_minutos_diarios, 2),
            "horas_diarias_requeridas": round(horas_diarias_requeridas, 2),
            "jornada_laboral_diaria": JORNADA_LABORAL_DIARIA,
            "factor_tiempo_neto_productivo": factor_tiempo_neto,
            "horas_netas_efectivas_por_persona": round(horas_netas_por_persona, 2),
            "numero_personas_requeridas": round(
                (total_minutos_diarios * (1 + tiempo_muerto) / 60) / horas_netas_por_persona, 3
            ),
            "tiempo_muerto": round(tiempo_muerto, 4),
            "horas_diarias_requeridas_final": round(
                total_minutos_diarios * (1 + tiempo_muerto) / 60, 2
            )
        }

    # -----------------------------------------------------------------------
    # 11. ANÁLISIS IA PARA EL ANALISTA
    # -----------------------------------------------------------------------
    def analisis_analista_ia(self, actividades, metricas, contexto):
        prompt = f"""
Eres un ANALISTA SENIOR en Levantamiento de Cargas Laborales y Dimensionamiento Operativo.
Tu tarea NO es recalcular datos, sino INTERPRETAR, VALIDAR COHERENCIA y EMITIR JUICIO PROFESIONAL
a partir de la información suministrada.

=========================
CONTEXTO ORGANIZACIONAL
=========================
Cargo: {contexto.get("cargo")}
Vicepresidencia: {contexto.get("vicepresidencia")}

=========================
ACTIVIDADES ANALIZADAS
=========================
{json.dumps(actividades, indent=2, ensure_ascii=False)}

=========================
MÉTRICAS CALCULADAS
=========================
{json.dumps(metricas, indent=2, ensure_ascii=False)}

=========================
INSTRUCCIONES DE ANÁLISIS
=========================

1. Analiza el NIVEL DE CARGA LABORAL.
2. Evalúa la COHERENCIA del resultado.
3. Interpreta el BALANCE PHVA.
4. Identifica RIESGOS OPERATIVOS.
5. Detecta OPORTUNIDADES DE AUTOMATIZACIÓN O MEJORA.
6. Formula RECOMENDACIONES EJECUTIVAS.

Menciona los valores numéricos mientras das el análisis e interpreta su significado.

Responde en español con lenguaje profesional, estructurado en los bloques anteriores.
"""
        reply = llm.invoke(prompt)
        return reply.content

    # -----------------------------------------------------------------------
    # 12. CAPTURAR CARGO Y VICEPRESIDENCIA (WIDGETS)
    # -----------------------------------------------------------------------
    def capturar_contexto_usuario(self):
        import ipywidgets as widgets
        from IPython.display import display

        cargo = widgets.Text(
            description="Cargo:",
            placeholder="Ej: Analista de Datos",
            style={'description_width': '120px'},
            layout=widgets.Layout(width='400px')
        )
        vicepresidencia = widgets.Text(
            description="Vicepresidencia:",
            placeholder="Ej: Tecnología",
            style={'description_width': '120px'},
            layout=widgets.Layout(width='400px')
        )
        display(cargo, vicepresidencia)
        return {"cargo": cargo, "vicepresidencia": vicepresidencia}

    # -----------------------------------------------------------------------
    # 13. OBTENER VALORES DE CONTEXTO
    # -----------------------------------------------------------------------
    def obtener_contexto_valores(self, contexto_widgets):
        return {
            "cargo": contexto_widgets["cargo"].value,
            "vicepresidencia": contexto_widgets["vicepresidencia"].value
        }

    # -----------------------------------------------------------------------
    # 14–15. CAPTURA DE INPUTS POR ACTIVIDAD (WIDGETS — se conserva para
    #         el flujo de actividades detectadas en la grabación)
    # -----------------------------------------------------------------------
    def capturar_inputs_usuario(self):
        import ipywidgets as widgets
        from IPython.display import display

        frecuencia = widgets.Dropdown(
            options=self.FRECUENCIAS_VALIDAS,
            description="Frecuencia:",
            style={'description_width': '120px'}
        )
        volumen = widgets.IntText(
            description="Volumen:",
            value=1,
            style={'description_width': '120px'}
        )
        autonomia = widgets.IntSlider(
            description="Autonomía (%):",
            min=0, max=100, value=80,
            style={'description_width': '120px'}
        )
        proceso_area = widgets.Textarea(
            description="Proceso del área:",
            layout=widgets.Layout(width='500px', height='80px'),
            style={'description_width': '120px'}
        )
        observaciones = widgets.Textarea(
            description="Observaciones:",
            layout=widgets.Layout(width='500px', height='80px'),
            style={'description_width': '120px'}
        )
        display(frecuencia, volumen, autonomia, proceso_area, observaciones)
        return {
            "frecuencia": frecuencia,
            "volumen": volumen,
            "proceso_area": proceso_area,
            "autonomia": autonomia,
            "observaciones": observaciones
        }

    def obtener_actividades_unicas(self, actividades):
        vistas = set()
        unicas = []
        for act in actividades:
            nombre = act["actividad"].strip().lower()
            if nombre not in vistas:
                vistas.add(nombre)
                unicas.append(act)
        return unicas

    def capturar_inputs_usuario_actividades_unicas(self, actividades):
        actividades_unicas = self.obtener_actividades_unicas(actividades)
        inputs_por_actividad = {}
        for act in actividades_unicas:
            print(f"\n Actividad: {act['actividad']}")
            inputs_por_actividad[act["actividad"].strip().lower()] = (
                self.capturar_inputs_usuario()
            )
        return inputs_por_actividad

    def obtener_inputs_usuario_valores(self, inputs_widgets):
        return {
            "frecuencia": inputs_widgets["frecuencia"].value,
            "volumen": inputs_widgets["volumen"].value,
            "autonomia": inputs_widgets["autonomia"].value,
            "proceso_area": inputs_widgets["proceso_area"].value,
            "observaciones": inputs_widgets["observaciones"].value
        }

    # -----------------------------------------------------------------------
    # 16. EDITAR ACTIVIDADES (GENERAL)
    # -----------------------------------------------------------------------
    def editar_actividades(self, actividades):
        while True:
            if not actividades:
                print("\n⚠️ No hay actividades para editar.")
                break

            print("\n✏️ ACTIVIDADES DISPONIBLES:")
            for i, act in enumerate(actividades, start=1):
                print(f"{i}. {act['nombre']}")

            opcion = input(
                "\nIngrese el número de la actividad "
                "(o 'salir' para terminar ajustes): "
            ).strip().lower()

            if opcion == "salir":
                break

            if not opcion.isdigit() or not (1 <= int(opcion) <= len(actividades)):
                print("⚠️ Opción inválida")
                continue

            idx = int(opcion) - 1
            actividad = actividades[idx]

            print(f"\n🔧 Actividad seleccionada: {actividad['nombre']}")
            accion = input(
                "¿Qué deseas hacer? (editar / eliminar / cancelar): "
            ).strip().lower()

            if accion == "cancelar":
                continue

            if accion == "eliminar":
                actividad_original = copy.deepcopy(actividad)
                confirmacion = input(
                    f"⚠️ ¿Seguro que deseas eliminar '{actividad['nombre']}'? (si / no): "
                ).strip().lower()
                if confirmacion == "si":
                    actividades.pop(idx)
                    print("🗑️ Actividad eliminada.")
                    confirmar_final = input(
                        "¿Confirmas la eliminación? (si / no): "
                    ).strip().lower()
                    if confirmar_final != "si":
                        actividades.insert(idx, actividad_original)
                        print("↩️ Eliminación revertida.")
                continue

            if accion != "editar":
                print("⚠️ Acción no válida.")
                continue

            campos_editables = [
                "frecuencia", "volumen", "duracion_min",
                "autonomia", "observaciones", "unidad_medida", "phva"
            ]
            print("\nCampos editables:")
            for c in campos_editables:
                print(f"- {c} (actual: {actividad.get(c)})")

            campo = input("\nCampo a modificar: ").strip()
            if campo not in campos_editables:
                print("⚠️ Campo no editable")
                continue

            nuevo_valor = input("Nuevo valor: ").strip()
            if campo in ("volumen", "duracion_min", "autonomia"):
                nuevo_valor = int(nuevo_valor)

            actividad_original = copy.deepcopy(actividad)
            actividad[campo] = nuevo_valor

            print("\n🧾 RESUMEN DEL CAMBIO:")
            print(f"- Actividad: {actividad['nombre']}")
            print(f"- Campo modificado: {campo}")
            print(f"- Valor anterior: {actividad_original.get(campo)}")
            print(f"- Nuevo valor: {nuevo_valor}")

            confirmar_cambio = input(
                "\n¿Confirmas este cambio? (si / no): "
            ).strip().lower()
            if confirmar_cambio != "si":
                actividades[idx] = actividad_original
                print("↩️ Cambio revertido.")
            else:
                print("✅ Cambio confirmado.")

        return actividades

    # -----------------------------------------------------------------------
    # 16b. EDITAR UNA ACTIVIDAD INDIVIDUAL (usada en confirmación por actividad)
    # -----------------------------------------------------------------------
    def _editar_actividad_individual(self, actividad):
        """
        Permite editar campos individuales de UNA actividad específica
        sin tener que reingresar toda la información.

        Se invoca cuando el usuario dice "no" en la confirmación por actividad.
        Devuelve la actividad (con los cambios aplicados).
        """
        campos_editables = [
            "nombre", "frecuencia", "volumen", "duracion_min",
            "autonomia", "proceso_area", "observaciones"
        ]

        print("\n" + "─" * 60)
        print("  ✏️  EDICIÓN DE CAMPOS — Actividad")
        print("─" * 60)

        while True:
            print("\nCampos disponibles para editar:")
            for c in campos_editables:
                print(f"  - {c}: {actividad.get(c, 'N/A')}")

            campo = input(
                "\nEscribe el campo que quieres modificar "
                "(o 'listo' para terminar): "
            ).strip().lower()

            if campo == "listo":
                break

            if campo not in campos_editables:
                print("⚠️ Campo no reconocido. Intenta de nuevo.")
                continue

            # Validaciones específicas por campo
            if campo == "frecuencia":
                opciones_str = "  ".join(
                    f"{i+1}) {f}" for i, f in enumerate(self.FRECUENCIAS_VALIDAS)
                )
                print(f"\n  Frecuencias válidas:\n  {opciones_str}")
                while True:
                    resp = input("Nueva frecuencia (número u opción): ").strip()
                    if resp.isdigit() and 1 <= int(resp) <= len(self.FRECUENCIAS_VALIDAS):
                        nuevo_valor = self.FRECUENCIAS_VALIDAS[int(resp) - 1]
                        break
                    elif resp.capitalize() in self.FRECUENCIAS_VALIDAS:
                        nuevo_valor = resp.capitalize()
                        break
                    else:
                        print(
                            f"⚠️ Opción no válida. Elige un número del "
                            f"1 al {len(self.FRECUENCIAS_VALIDAS)}."
                        )

            elif campo in ("volumen", "duracion_min"):
                while True:
                    resp = input(f"Nuevo valor para '{campo}' (entero > 0): ").strip()
                    if resp.isdigit() and int(resp) > 0:
                        nuevo_valor = int(resp)
                        break
                    print("⚠️ Ingresa un número entero mayor a 0.")

            elif campo == "autonomia":
                autonomia_opciones = list(range(5, 105, 5))
                fila = "  " + "  ".join(f"{p}%" for p in autonomia_opciones)
                print(f"\n  Opciones de autonomía:\n{fila}")
                while True:
                    resp = input("Nueva autonomía (%): ").strip().replace("%", "").strip()
                    if resp.isdigit() and int(resp) in autonomia_opciones:
                        nuevo_valor = int(resp)
                        break
                    print("⚠️ Elige un valor de 5 en 5 entre 5% y 100%.")

            else:
                nuevo_valor = input(f"Nuevo valor para '{campo}': ").strip()

            valor_anterior = actividad.get(campo)
            actividad[campo] = nuevo_valor

            print(f"\n🧾 Cambio aplicado:")
            print(f"  Campo     : {campo}")
            print(f"  Antes     : {valor_anterior}")
            print(f"  Después   : {nuevo_valor}")

            confirmar = input("¿Confirmas este cambio? (si / no): ").strip().lower()
            if confirmar != "si":
                actividad[campo] = valor_anterior
                print("↩️ Cambio revertido.")
            else:
                print("✅ Cambio confirmado.")

        return actividad

    # -----------------------------------------------------------------------
    # 17. FUSIÓN ACTIVIDADES NO DIARIAS
    # -----------------------------------------------------------------------
    def fusionar_inputs_usuario_nodiarias(
        self, actividades_base, actividades_enriquecidas
    ):
        actividades_finales_nodiarias = []
        for base, ia in zip(actividades_base, actividades_enriquecidas):
            actividades_finales_nodiarias.append({
                "nombre": base["nombre"],
                "duracion_min": base["duracion_min"],
                "unidad_medida": ia["unidad_medida"],
                "phva": ia["phva"],
                "frecuencia": base["frecuencia"],
                "volumen": base["volumen"],
                "proceso_area": base["proceso_area"],
                "autonomia": base["autonomia"],
                "observaciones": base.get("observaciones", "")
            })
        return actividades_finales_nodiarias

    # -----------------------------------------------------------------------
    # 18. VALIDACIÓN DE COHERENCIA GLOBAL
    # -----------------------------------------------------------------------
    def validar_coherencia_global(self, actividades, contexto):
        prompt = f"""
Actúa como un validador técnico de coherencia de actividades laborales.

El volumen representa el número de veces que se ejecuta la actividad
dentro del periodo definido por la frecuencia.

Tu tarea:
1. Detectar únicamente incoherencias internas.
2. No hacer juicios del rol ni usar información externa.

Debes validar únicamente:
- volumen × duración dentro del periodo de la frecuencia
- acumulaciones problemáticas dentro del mismo periodo
- duraciones extremadamente bajas o altas en relación con el volumen
- inconsistencias temporales si existen

⚠️ IMPORTANTE:
- Solo muestra las actividades donde detectes incoherencias.
- NO muestres actividades que estén correctas.

Formato de salida obligatorio:

------------------------------------------------------------
🔎 VALIDACIÓN DE COHERENCIA

Para cada actividad con incoherencia:

📌 Actividad: [Nombre exacto]

Campos evaluados:
- Frecuencia: [valor]
- Volumen: [valor]
- Duración por unidad: [valor + unidad]

Cálculo implícito:
[volumen × duración = total dentro del periodo]

Incoherencia detectada:
- [Descripción técnica breve]

------------------------------------------------------------

🧾 Resumen general:
- Total de actividades analizadas: [número]
- Actividades con incoherencias: [número]
- Errores matemáticos explícitos: [sí/no]

Al final devuelve SIEMPRE un bloque JSON así:

{{
    "actividades_incoherentes": ["Nombre 1", "Nombre 2"]
}}

Si no hay incoherencias:

{{
    "actividades_incoherentes": []
}}

=========================
CONTEXTO
=========================
Cargo: {contexto.get("cargo")}
Vicepresidencia: {contexto.get("vicepresidencia")}

=========================
ACTIVIDADES REGISTRADAS
=========================
{json.dumps(actividades, indent=2, ensure_ascii=False)}

Responde en español.
"""
        reply = llm.invoke(prompt)
        content = reply.content
        actividades_incoherentes = []
        try:
            json_start = content.rfind("{")
            if json_start != -1:
                json_text = content[json_start:]
                json_data = json.loads(json_text)
                actividades_incoherentes = json_data.get(
                    "actividades_incoherentes", []
                )
        except Exception:
            actividades_incoherentes = []
        return content, actividades_incoherentes

    # -----------------------------------------------------------------------
    # 18.1 EDITAR ACTIVIDADES INCOHERENTES
    # -----------------------------------------------------------------------
    def editar_actividades_incoherentes(self, actividades, actividades_incoherentes):
        actividades_filtradas = [
            act for act in actividades
            if act["nombre"] in actividades_incoherentes
        ]
        if not actividades_filtradas:
            print("\n✅ No hay actividades con incoherencias para editar.")
            return actividades

        while True:
            print("\n⚠️ ACTIVIDADES CON INCOHERENCIAS:")
            for i, act in enumerate(actividades_filtradas, start=1):
                print(f"{i}. {act['nombre']}")

            opcion = input(
                "\nSeleccione el número de la actividad "
                "(o 'salir' para terminar ajustes): "
            ).strip().lower()

            if opcion == "salir":
                break

            if not opcion.isdigit() or not (
                1 <= int(opcion) <= len(actividades_filtradas)
            ):
                print("⚠️ Opción inválida")
                continue

            actividad = actividades_filtradas[int(opcion) - 1]
            idx = next(
                i for i, a in enumerate(actividades)
                if a["nombre"] == actividad["nombre"]
            )

            print(f"\n🔧 Actividad seleccionada: {actividad['nombre']}")
            accion = input(
                "¿Qué deseas hacer? (editar / eliminar / cancelar): "
            ).strip().lower()

            if accion == "cancelar":
                continue

            if accion == "eliminar":
                confirmacion = input(
                    f"¿Confirmas eliminar '{actividad['nombre']}'? (si / no): "
                ).strip().lower()
                if confirmacion == "si":
                    actividades.pop(idx)
                    print("🗑️ Actividad eliminada.")
                continue

            if accion != "editar":
                print("⚠️ Acción no válida.")
                continue

            campos_editables = [
                "frecuencia", "volumen", "duracion_min",
                "autonomia", "observaciones", "unidad_medida", "phva"
            ]
            print("\nCampos editables:")
            for c in campos_editables:
                print(f"- {c} (actual: {actividad.get(c)})")

            campo = input("\nCampo a modificar: ").strip()
            if campo not in campos_editables:
                print("⚠️ Campo no editable")
                continue

            nuevo_valor = input("Nuevo valor: ").strip()
            if campo in ("volumen", "duracion_min", "autonomia"):
                nuevo_valor = int(nuevo_valor)

            actividad_original = copy.deepcopy(actividad)
            actividades[idx][campo] = nuevo_valor

            print("\n🧾 RESUMEN DEL CAMBIO:")
            print(f"- Actividad: {actividad['nombre']}")
            print(f"- Campo: {campo}")
            print(f"- Antes: {actividad_original.get(campo)}")
            print(f"- Después: {nuevo_valor}")

            confirmar = input("\n¿Confirmas el cambio? (si / no): ").strip().lower()
            if confirmar != "si":
                actividades[idx] = actividad_original
                print("↩️ Cambio revertido.")
            else:
                print("✅ Cambio confirmado.")

        return actividades

    # -----------------------------------------------------------------------
    # 19. RESUMEN AUTOMÁTICO DE CAMBIOS
    # -----------------------------------------------------------------------
    def generar_resumen_cambios(self, antes, despues):
        prompt = f"""
Eres un asistente que compara versiones de información.

REGLAS:
- No recalcules métricas.
- No hagas análisis.
- Solo describe qué cambió.
- Sé breve y claro.
- Si no hubo cambios, indícalo.

====================
ANTES
====================
{json.dumps(antes, indent=2, ensure_ascii=False)}

====================
DESPUÉS
====================
{json.dumps(despues, indent=2, ensure_ascii=False)}

Responde en español en formato claro y organizado.
"""
        reply = llm.invoke(prompt)
        return reply.content

    # =======================================================================
    # ★ FLUJO CONVERSACIONAL PARA ACTIVIDADES NO DIARIAS
    # =======================================================================

    # -----------------------------------------------------------------------
    # A. VALIDAR UNA ACTIVIDAD CON IA (coherencia individual)
    # -----------------------------------------------------------------------
    def _validar_coherencia_actividad_ia(self, actividad, min_disponibles=None):
        tope_diario = round(min_disponibles, 1) if min_disponibles is not None else 510
        tope_label  = (
            f"{tope_diario} min disponibles en la jornada "
            f"(descontando actividades ya registradas)"
            if min_disponibles is not None
            else "510 min (8.5 h, jornada completa)"
        )

        prompt = f"""
Actúa como validador técnico de coherencia de actividades laborales.

Tienes UNA actividad con los siguientes datos:
{json.dumps(actividad, indent=2, ensure_ascii=False)}

Definición clave:
- "volumen" = número de veces que se ejecuta la actividad dentro del
  periodo definido por "frecuencia".
- "duracion_min" = minutos que tarda UNA ejecución.

Tiempo disponible en la jornada diaria para validar: {tope_label}

Debes verificar:
1. ¿El volumen × duración_min supera el tiempo disponible en el periodo?
   Usa el tiempo disponible indicado arriba como tope para frecuencia Diaria.
   Para otras frecuencias multiplica por los días del periodo:
   - Semanal: {tope_diario} × 5
   - Quincenal: {tope_diario} × 10
   - Mensual: {tope_diario} × 21
   - Bimensual: {tope_diario} × 42
   - Trimestral: {tope_diario} × 63
   - Semestral: {tope_diario} × 126
   - Anual: {tope_diario} × 252
2. ¿La duración individual parece extremadamente corta (< 1 min) o
   larga (> 480 min) para el tipo de actividad?
3. ¿Hay alguna otra inconsistencia lógica evidente?

Si hay incoherencia, genera recomendaciones CONCRETAS y ACCIONABLES sobre
qué valores específicos cambiar (campo, valor actual → valor sugerido).

Responde EXCLUSIVAMENTE con un JSON válido, sin texto adicional:

{{
  "hay_incoherencia": true | false,
  "explicacion": "descripción breve y clara de la incoherencia, o cadena vacía si no hay",
  "recomendaciones": [
    "Recomendación concreta 1 (campo: valor actual → valor sugerido)",
    "Recomendación concreta 2 (si aplica)"
  ]
}}

Si no hay incoherencia, devuelve:
{{
  "hay_incoherencia": false,
  "explicacion": "",
  "recomendaciones": []
}}
"""
        reply = llm.invoke(prompt)
        try:
            resultado = self._parse_json_seguro(reply.content)
            return (
                resultado.get("hay_incoherencia", False),
                resultado.get("explicacion", ""),
                resultado.get("recomendaciones", []),
            )
        except Exception:
            return False, "", []

    # -----------------------------------------------------------------------
    # B. MOSTRAR INCOHERENCIA Y PERMITIR CORRECCIÓN INMEDIATA
    # -----------------------------------------------------------------------
    def _mostrar_incoherencia_y_corregir(
        self, actividad, explicacion, recomendaciones, min_disponibles=None
    ):
        """
        Muestra la incoherencia detectada y las recomendaciones del LLM.

        Flujo simplificado:
          1. Muestra la explicación y las recomendaciones.
          2. Pregunta si acepta las correcciones sugeridas o conserva los datos.
             - "si"  → aplica automáticamente los cambios de las recomendaciones
                       (parsea campo y valor de cada recomendación) y re-valida.
                       Si aún hay incoherencia, muestra el nuevo resultado y
                       vuelve a ofrecer aceptar/conservar.
             - "no"  → conserva la actividad tal como está y continúa.
        """
        # ── Helper: parsear recomendaciones del LLM ────────────────────────
        # El LLM devuelve líneas como:
        #   "duracion_min: 30 → 150 (ajustar ...)"
        #   "volumen: 3 → 2 (considerar ...)"
        # Se extrae el campo y el nuevo valor numérico o textual.
        def _aplicar_recomendaciones(act, recs):
            """
            Parsea cada recomendación e intenta aplicarla a la actividad.
            Devuelve la actividad modificada y una lista de cambios aplicados.
            """
            import re as _re
            campos_numericos = {"volumen", "duracion_min", "autonomia"}
            cambios = []

            for rec in recs:
                # Buscar patrón: "campo: valor_actual → nuevo_valor"
                m = _re.search(
                    r"([a-zA-Z_]+)\s*:\s*[\w\s,.\[\]]+?→\s*([\w.]+)",
                    rec
                )
                if not m:
                    continue
                campo     = m.group(1).strip().lower()
                val_str   = m.group(2).strip()

                # Solo campos editables reconocidos
                campos_validos = {
                    "volumen", "duracion_min", "autonomia",
                    "frecuencia", "proceso_area", "observaciones"
                }
                if campo not in campos_validos:
                    continue

                # Conversión de tipo
                if campo in campos_numericos:
                    try:
                        nuevo_valor = int(float(val_str))
                    except ValueError:
                        continue
                    # Validación autonomía
                    if campo == "autonomia" and nuevo_valor not in range(5, 105, 5):
                        continue
                elif campo == "frecuencia":
                    val_cap = val_str.capitalize()
                    if val_cap not in self.FRECUENCIAS_VALIDAS:
                        continue
                    nuevo_valor = val_cap
                else:
                    nuevo_valor = val_str

                valor_anterior = act.get(campo)
                act[campo] = nuevo_valor
                cambios.append((campo, valor_anterior, nuevo_valor))

            return act, cambios

        # ── Bucle principal ────────────────────────────────────────────────
        expl_actual = explicacion
        recs_actual = recomendaciones

        while True:
            print("\n" + "─" * 60)
            print("⚠️  POSIBLE INCOHERENCIA DETECTADA")
            print("─" * 60)
            print(f"   {expl_actual}")

            if recs_actual:
                print("\n💡 Correcciones sugeridas:")
                for i, rec in enumerate(recs_actual, start=1):
                    print(f"   {i}. {rec}")
            print("─" * 60)

            while True:
                respuesta = input(
                    "\n¿Aceptas las correcciones sugeridas? (si / no): "
                ).strip().lower()
                if respuesta in ("si", "no"):
                    break
                print("⚠️ Por favor escribe 'si' o 'no'.")

            if respuesta != "si":
                print("✅ Se conserva la información tal como fue ingresada.")
                break

            # Aplicar las correcciones automáticamente
            actividad, cambios = _aplicar_recomendaciones(actividad, recs_actual)

            if cambios:
                print("\n🧾 Cambios aplicados:")
                for campo, antes, despues in cambios:
                    print(f"   • {campo}: {antes} → {despues}")
            else:
                print(
                    "\nℹ️  No se pudieron aplicar las correcciones automáticamente.\n"
                    "   Puedes ajustar los datos en la pantalla de confirmación."
                )
                break

            # Re-validar con los nuevos valores
            print("\n🔍 Re-validando coherencia...")
            hay_inc, expl_nuevo, recs_nuevo = self._validar_coherencia_actividad_ia(
                actividad, min_disponibles=min_disponibles
            )

            if not hay_inc:
                print("✅ Los datos son coherentes ahora.")
                break

            # Sigue habiendo incoherencia → mostrar de nuevo y preguntar
            expl_actual = expl_nuevo
            recs_actual = recs_nuevo

        return actividad

    # -----------------------------------------------------------------------
    # C1. CALCULAR MÉTRICAS PARCIALES ACUMULADAS
    # -----------------------------------------------------------------------
    def _calcular_metricas_parciales(self, actividades_hasta_ahora):
        DIAS_LABORALES_MES   = 21
        MINUTOS_JORNADA_MES  = 220 * 60
        JORNADA_DIARIA       = 8.5

        FACTOR_FRECUENCIA = {
            "Diario":     1,
            "Semanal":    1 / 5,
            "Quincenal":  1 / 10,
            "Mensual":    1 / 21,
            "Bimensual":  1 / 42,
            "Trimestral": 1 / 63,
            "Semestral":  1 / 126,
            "Anual":      1 / 252,
        }

        tiempo_muerto = sum([
            (1 / 8)        / JORNADA_DIARIA,
            ((3 * 5) / 60) / JORNADA_DIARIA,
            (15 / 60)      / JORNADA_DIARIA,
            (10 / 60)      / JORNADA_DIARIA,
            (20 / 60)      / JORNADA_DIARIA,
        ])
        factor_neto = 1 - tiempo_muerto
        horas_netas_por_persona = JORNADA_DIARIA * factor_neto

        total_min_diarios          = 0.0
        total_carga_w_sin_tm       = 0.0
        total_carga_individual     = 0.0
        metricas_por_actividad     = []

        for act in actividades_hasta_ahora:
            factor    = FACTOR_FRECUENCIA.get(act.get("frecuencia", ""), 0)
            min_dia   = act["duracion_min"] * act["volumen"] * factor
            min_mes   = min_dia * DIAS_LABORALES_MES
            carga_w   = min_mes / MINUTOS_JORNADA_MES
            carga_ind = carga_w * (act["autonomia"] / 100)

            total_min_diarios      += min_dia
            total_carga_w_sin_tm   += carga_w
            total_carga_individual += carga_ind

            metricas_por_actividad.append({
                "nombre":           act["nombre"],
                "frecuencia":       act.get("frecuencia"),
                "min_diarios":      round(min_dia,  2),
                "carga_w_sin_tm":   round(carga_w,  4),
                "carga_individual": round(carga_ind, 4),
            })

        horas_diarias_requeridas = total_min_diarios / 60
        horas_con_tm = total_min_diarios * (1 + tiempo_muerto) / 60
        personas_req = horas_con_tm / horas_netas_por_persona if horas_netas_por_persona else 0

        return {
            "total_min_diarios":        round(total_min_diarios, 2),
            "horas_diarias_requeridas": round(horas_diarias_requeridas, 2),
            "horas_con_tiempo_muerto":  round(horas_con_tm, 2),
            "total_carga_w_sin_tm":     round(total_carga_w_sin_tm, 4),
            "total_carga_individual":   round(total_carga_individual, 4),
            "personas_requeridas":      round(personas_req, 3),
            "jornada_laboral_diaria":   JORNADA_DIARIA,
            "horas_netas_por_persona":  round(horas_netas_por_persona, 2),
            "factor_tiempo_neto":       round(factor_neto, 4),
            "num_actividades":          len(actividades_hasta_ahora),
            "metricas_por_actividad":   metricas_por_actividad,
        }

    # -----------------------------------------------------------------------
    # C2. ANÁLISIS CONTEXTUAL POR ACTIVIDAD (LLM)
    # -----------------------------------------------------------------------
    def _analisis_actividad_en_contexto(
        self, actividad_nueva, actividades_previas, metricas, contexto
    ):
        prompt = f"""
Eres un analista experto en Levantamiento de Cargas Laborales.
Tu rol en este momento es acompañar al colaborador durante el registro
de sus actividades y darle retroalimentación útil tras registrar cada una.

=========================
CONTEXTO DEL COLABORADOR
=========================
Cargo           : {contexto.get("cargo")}
Vicepresidencia : {contexto.get("vicepresidencia")}

=========================
ACTIVIDAD RECIÉN REGISTRADA
=========================
{json.dumps(actividad_nueva, indent=2, ensure_ascii=False)}

=========================
ACTIVIDADES YA REGISTRADAS ANTES DE ESTA
=========================
{json.dumps(actividades_previas, indent=2, ensure_ascii=False) if actividades_previas else "Ninguna (esta es la primera actividad registrada)"}

=========================
MÉTRICAS INTERNAS CALCULADAS (para tu análisis — NO las expongas)
=========================
{json.dumps(metricas, indent=2, ensure_ascii=False)}

=========================
INSTRUCCIONES ESTRICTAS
=========================
1. Usa las métricas internas SOLO como base de tu interpretación.
   NO menciones ni repitas los valores numéricos de carga laboral,
   dotación, minutos diarios ni factores de tiempo al usuario.

2. Tu análisis debe cubrir EXACTAMENTE estos dos puntos:
   a) ¿Qué representa esta actividad para la carga del colaborador?
   b) ¿Cómo se ve la carga acumulada hasta ahora en relación con
      una jornada laboral normal?

3. Si es la primera actividad, solo analiza el punto (a).

4. Cierra con una frase breve y motivadora que invite al colaborador
   a continuar con el registro.

5. NO hagas preguntas. NO solicites correcciones.

Responde en español, tono profesional pero cercano.
"""
        reply = llm.invoke(prompt)
        return reply.content

    # -----------------------------------------------------------------------
    # C. MOSTRAR RESUMEN DE UNA ACTIVIDAD Y PEDIR CONFIRMACIÓN
    #    (MODIFICADO v5: si el usuario dice "no", entra al editor de campos)
    # -----------------------------------------------------------------------
    def _confirmar_actividad(self, actividad, numero, actividades_previas, contexto):
        """
        1. Calcula métricas parciales acumuladas (uso interno / analista).
        2. Muestra el resumen de la actividad al usuario.
        3. Pide confirmación.
           - Si dice "no": abre el editor de campos individuales
             (sin tener que reingresar todo desde cero).

        Devuelve True cuando el usuario confirma la actividad, False si
        después de editar quiere volver a revisar desde el inicio del paso.
        """
        todas    = actividades_previas + [actividad]
        metricas = self._calcular_metricas_parciales(todas)

        analisis_interno = self._analisis_actividad_en_contexto(
            actividad_nueva     = actividad,
            actividades_previas = actividades_previas,
            metricas            = metricas,
            contexto            = contexto,
        )
        actividad["_analisis_interno"] = analisis_interno

        while True:
            # --- Resumen visible para el colaborador ---
            print("\n" + "═" * 60)
            print(f"  RESUMEN — Actividad {numero}")
            print("═" * 60)
            print(f"  Nombre            : {actividad.get('nombre')}")
            if actividad.get("descripcion"):
                print(f"  Descripción       : {actividad.get('descripcion')}")
            print(f"  Frecuencia        : {actividad.get('frecuencia')}")
            print(f"  Volumen           : {actividad.get('volumen')} "
                  f"vez/veces por periodo")
            print(f"  Duración          : {actividad.get('duracion_min')} min "
                  f"por ejecución")
            print(f"  Autonomía         : {actividad.get('autonomia')}%")
            print(f"  Proceso del área  : {actividad.get('proceso_area') or '—'}")
            if actividad.get("observaciones"):
                print(f"  Observaciones     : {actividad.get('observaciones')}")
            print("═" * 60)

            resp = input(
                "\n¿Esta información es correcta? (si / no): "
            ).strip().lower()

            if resp == "si":
                return True

            # --- El usuario quiere corregir: editor de campos individuales ---
            print(
                "\n🔄 Puedes modificar los campos que necesites "
                "sin tener que empezar de cero."
            )
            actividad = self._editar_actividad_individual(actividad)
            print("\n✅ Cambios aplicados. Revisando el resumen actualizado...")

    # -----------------------------------------------------------------------
    # D. RECOGER UNA ACTIVIDAD COMPLETA — incluye selección de proceso del área
    # -----------------------------------------------------------------------
    def _preguntar_proceso_area(self, procesos_registrados):
        """
        Muestra los procesos del área ya registrados (si los hay) para que
        el usuario pueda seleccionar uno existente o ingresar uno nuevo.

        Parámetros
        ----------
        procesos_registrados : list[str] — procesos únicos ya capturados

        Retorna
        -------
        str — proceso del área seleccionado o ingresado
        """
        procesos_unicos = list(dict.fromkeys(
            p for p in procesos_registrados if p
        ))  # preserva orden, elimina duplicados y vacíos

        if procesos_unicos:
            print("\n¿A qué proceso del área pertenece esta actividad?")
            print("  Procesos ya registrados:")
            for i, proc in enumerate(procesos_unicos, start=1):
                print(f"    {i}) {proc}")
            print(f"    {len(procesos_unicos) + 1}) Ingresar un proceso nuevo")

            while True:
                resp = input(
                    f"Selecciona un número del 1 al "
                    f"{len(procesos_unicos) + 1}: "
                ).strip()

                if resp.isdigit():
                    opcion = int(resp)
                    if 1 <= opcion <= len(procesos_unicos):
                        return procesos_unicos[opcion - 1]
                    elif opcion == len(procesos_unicos) + 1:
                        # Nuevo proceso
                        break
                    else:
                        print(
                            f"⚠️ Elige un número del 1 al "
                            f"{len(procesos_unicos) + 1}."
                        )
                else:
                    print("⚠️ Por favor, ingresa un número.")

        # Sin procesos previos o el usuario quiere uno nuevo
        proceso = ""
        while not proceso:
            proceso = input(
                "\n¿A qué proceso del área pertenece esta actividad? "
                "(Ej: Gestión de proveedores, Reportes, Atención al cliente): "
            ).strip()
            if not proceso:
                print("⚠️ Por favor, ingresa el proceso del área.")
        return proceso

    def _preguntar_actividad(self, numero, procesos_registrados=None):
        """
        Conduce el diálogo para capturar todos los campos de
        una actividad y devuelve el diccionario con los datos.

        CAMBIOS v5:
        - proceso_area usa _preguntar_proceso_area() para reutilizar
          procesos ya registrados.
        """
        if procesos_registrados is None:
            procesos_registrados = []

        print("\n" + "─" * 60)
        print(f"  📋  ACTIVIDAD {numero}")
        print("─" * 60)

        # --- Nombre ---
        nombre = ""
        while not nombre:
            nombre = input(
                f"\n¿Cuál es la actividad {numero}? "
                "(describe brevemente qué haces): "
            ).strip()
            if not nombre:
                print("⚠️ Por favor, ingresa una descripción.")

        # --- Frecuencia ---
        frecuencia = None
        opciones_str = "  ".join(
            f"{i+1}) {f}" for i, f in enumerate(self.FRECUENCIAS_VALIDAS)
        )
        print(f"\n¿Con qué frecuencia realizas esta actividad?")
        print(f"  {opciones_str}")
        while frecuencia is None:
            resp = input("Selecciona el número o escribe la frecuencia: ").strip()
            if resp.isdigit() and 1 <= int(resp) <= len(self.FRECUENCIAS_VALIDAS):
                frecuencia = self.FRECUENCIAS_VALIDAS[int(resp) - 1]
            elif resp.capitalize() in self.FRECUENCIAS_VALIDAS:
                frecuencia = resp.capitalize()
            else:
                print(
                    "⚠️ Opción no válida. "
                    f"Elige un número del 1 al {len(self.FRECUENCIAS_VALIDAS)}."
                )

        # --- Volumen ---
        volumen = None
        print(
            f"\n¿Cuántas veces realizas esta actividad por periodo "
            f"({frecuencia.lower()})?"
        )
        print("  (Ej: si es Diario y la haces 3 veces al día → escribe 3)")
        while volumen is None:
            resp = input("Volumen: ").strip()
            if resp.isdigit() and int(resp) > 0:
                volumen = int(resp)
            else:
                print("⚠️ Ingresa un número entero mayor a 0.")

        # --- Duración ---
        duracion_min = None
        print(
            "\n¿Cuántos minutos te toma completar UNA ejecución "
            "de esta actividad?"
        )
        while duracion_min is None:
            resp = input("Duración (minutos): ").strip()
            if resp.isdigit() and int(resp) > 0:
                duracion_min = int(resp)
            else:
                print("⚠️ Ingresa un número entero mayor a 0.")

        # --- Autonomía (5% a 100% de 5 en 5) ---
        autonomia_opciones = list(range(5, 105, 5))
        autonomia = None
        print("\n¿Qué porcentaje de esta actividad recae sobre ti?")
        fila = "  " + "  ".join(f"{p}%" for p in autonomia_opciones)
        print(f"  Opciones disponibles (de 5 en 5):\n{fila}")
        print(
            "  (Ej: si eres el único responsable → 100%  |  "
            "si compartes al 50% con otro → 50%)"
        )
        while autonomia is None:
            resp = input("Autonomía (%): ").strip().replace("%", "").strip()
            if resp.isdigit() and int(resp) in autonomia_opciones:
                autonomia = int(resp)
            else:
                print(
                    "⚠️ Elige un valor de 5 en 5 entre 5% y 100%. "
                    "Ej: 5, 10, 15, ... , 95, 100."
                )

        # --- Proceso del área (con reutilización) ---
        proceso_area = self._preguntar_proceso_area(procesos_registrados)

        # --- Observaciones ---
        observaciones = input(
            "\n¿Tienes alguna observación adicional sobre esta actividad? "
            "(presiona Enter si no): "
        ).strip()

        return {
            "nombre":        nombre,
            "frecuencia":    frecuencia,
            "volumen":       volumen,
            "duracion_min":  duracion_min,
            "autonomia":     autonomia,
            "proceso_area":  proceso_area,
            "observaciones": observaciones,
        }

    # -----------------------------------------------------------------------
    # E.0  HELPERS RÁPIDOS (sin LLM)
    # -----------------------------------------------------------------------
    def _calcular_minutos_diarios_acumulados(self, actividades):
        FACTOR_FRECUENCIA = {
            "Diario": 1, "Semanal": 1/5, "Quincenal": 1/10,
            "Mensual": 1/21, "Bimensual": 1/42, "Trimestral": 1/63,
            "Semestral": 1/126, "Anual": 1/252,
        }
        total = 0.0
        for act in actividades:
            factor = FACTOR_FRECUENCIA.get(act.get("frecuencia", ""), 0)
            total += act["duracion_min"] * act["volumen"] * factor
        return round(total, 2)

    def _es_duplicada(self, nombre_nuevo, actividades_existentes, umbral=0.75):
        from difflib import SequenceMatcher
        nombre_norm = nombre_nuevo.strip().lower()
        for act in actividades_existentes:
            existente_norm = act["nombre"].strip().lower()
            ratio = SequenceMatcher(None, nombre_norm, existente_norm).ratio()
            if ratio >= umbral:
                return True, act["nombre"]
        return False, None

    # -----------------------------------------------------------------------
    # E. FLUJO PRINCIPAL CONVERSACIONAL
    #    MODIFICADO v5:
    #    - _preguntar_actividad recibe la lista de procesos registrados
    #    - Detecta relevancia/detalle con IA tras capturar la actividad
    #    - La pregunta de "otra actividad" solo acepta si/no
    # -----------------------------------------------------------------------
    def _capturar_actividades_conversacional_con_procesos(
        self, contexto, procesos_iniciales=None
    ):
        """
        Versión interna de capturar_actividades_conversacional que acepta
        una lista de procesos ya conocidos al inicio (por ejemplo, los que
        vienen de actividades completadas desde la transcripción de Teams).
        Esto permite que el selector de "proceso del área" muestre desde
        el principio los procesos ya registrados.
        """
        return self.capturar_actividades_conversacional(
            contexto, procesos_iniciales=procesos_iniciales or []
        )

    def capturar_actividades_conversacional(self, contexto, procesos_iniciales=None):
        """
        Conduce al usuario a través de un diálogo para registrar todas
        sus actividades.

        Parámetros
        ----------
        contexto          : dict  — {"cargo": ..., "vicepresidencia": ...}
        procesos_iniciales: list  — procesos del área ya conocidos al inicio
                                    (ej: los que vienen de actividades VTT
                                    ya completadas). Se ofrecen como opciones
                                    desde la primera actividad.
        """
        if procesos_iniciales is None:
            procesos_iniciales = []

        JORNADA_MIN = 8.5 * 60

        print("\n" + "═" * 60)
        print("  🗂️  REGISTRO DE ACTIVIDADES — MODO CONVERSACIONAL")
        print("═" * 60)
        print(
            "\nA continuación te haré preguntas sobre cada actividad que realizas.\n"
            "Puedes registrar todas las que quieras.\n"
            "Cuando termines, escribe 'no' cuando te pregunte si hay otra actividad."
        )

        actividades = []
        numero = 1
        tope_alcanzado = False

        while True:
            # ── Capturar + validar + confirmar ────────────────────────────
            while True:
                # Lista de procesos ya registrados para reutilizar
                # (incluye los iniciales provenientes de actividades VTT)
                procesos_registrados = list(procesos_iniciales) + [
                    act.get("proceso_area", "") for act in actividades
                ]

                actividad = self._preguntar_actividad(
                    numero,
                    procesos_registrados=procesos_registrados
                )

                # ── 1. Detección de duplicados ────────────────────────────
                es_dup, nombre_similar = self._es_duplicada(
                    actividad["nombre"], actividades
                )
                if es_dup:
                    print("\n" + "─" * 60)
                    print("⚠️  POSIBLE ACTIVIDAD DUPLICADA")
                    print("─" * 60)
                    print(
                        f"   La actividad \"{actividad['nombre']}\" es muy similar\n"
                        f"   a \"{nombre_similar}\", que ya fue registrada."
                    )
                    decision = input(
                        "\n   ¿Es realmente una actividad diferente? (si / no): "
                    ).strip().lower()
                    if decision != "si":
                        print("🔄 Descartada. Ingresa una actividad distinta.\n")
                        continue

                # ── 2. Evaluar relevancia y necesidad de detalle con IA ───
                print("\n🔍 Evaluando relevancia y completitud de la actividad...")
                evaluacion = self._evaluar_relevancia_y_detalle(
                    actividad,
                    contexto.get("cargo", ""),
                    contexto.get("vicepresidencia", "")
                )
                accion, actividad = self._manejar_alerta_relevancia(
                    actividad, evaluacion
                )
                if accion == "descartar":
                    print("🔄 Ingresa una nueva actividad.\n")
                    continue

                # ── 3. Calcular minutos disponibles restantes ─────────────
                min_acum_previo = self._calcular_minutos_diarios_acumulados(actividades)
                min_disponibles = max(JORNADA_MIN - min_acum_previo, 0)

                # ── 4. Validar coherencia con tope real de minutos ─────────
                print("\n🔍 Revisando coherencia de los datos ingresados...")
                hay_incoherencia, explicacion, recomendaciones = (
                    self._validar_coherencia_actividad_ia(
                        actividad,
                        min_disponibles=min_disponibles if min_acum_previo > 0 else None,
                    )
                )

                if hay_incoherencia:
                    actividad = self._mostrar_incoherencia_y_corregir(
                        actividad,
                        explicacion,
                        recomendaciones,
                        min_disponibles=min_disponibles if min_acum_previo > 0 else None,
                    )
                else:
                    print("✅ Coherencia OK — los datos ingresados son consistentes.")

                # ── 5. Confirmar actividad (con editor si dice "no") ───────
                confirmada = self._confirmar_actividad(
                    actividad           = actividad,
                    numero              = numero,
                    actividades_previas = actividades,
                    contexto            = contexto,
                )
                if confirmada:
                    actividades.append(actividad)
                    print(f"\n✅ Actividad {numero} registrada.")
                    break
                # Si _confirmar_actividad devuelve False (ya no debería ocurrir
                # por el nuevo flujo interno, pero se conserva por seguridad)
                else:
                    print(
                        f"\n🔄 Volvemos a registrar la actividad {numero}. "
                        "Ingresa los datos de nuevo.\n"
                    )

            # ── Barra de carga y alerta por minutos restantes ─────────────
            min_acum   = self._calcular_minutos_diarios_acumulados(actividades)
            horas_acum = min_acum / 60
            min_rest   = max(JORNADA_MIN - min_acum, 0)
            horas_rest = min_rest / 60

            pct     = min(min_acum / JORNADA_MIN, 1.0)
            bloques = int(pct * 20)
            barra   = "█" * bloques + "░" * (20 - bloques)
            print(
                f"\n  ⏱️  Carga acumulada: [{barra}] "
                f"{horas_acum:.1f} h de 8.5 h hábiles  "
                f"({horas_rest:.1f} h disponibles)"
            )

            if min_acum >= JORNADA_MIN and not tope_alcanzado:
                tope_alcanzado = True
                print("\n" + "⚠️ " * 20)
                print(
                    "  ATENCIÓN: Las actividades registradas ya ocupan\n"
                    "  TODA la jornada laboral disponible (8.5 horas).\n"
                    "  Si agregas más, la carga superará la capacidad diaria."
                )
                print("⚠️ " * 20)
                continuar = input(
                    "\n  ¿Deseas seguir registrando actividades de todas formas? "
                    "(si / no): "
                ).strip().lower()
                if continuar != "si":
                    break

            elif min_acum >= JORNADA_MIN and tope_alcanzado:
                exceso_min = round(min_acum - JORNADA_MIN, 1)
                print(
                    f"  ⚠️  Ya superaste la jornada diaria en "
                    f"~{exceso_min} min equivalentes."
                )

            # ── ¿Hay más actividades? (SOLO acepta si/no) ─────────────────
            print("\n" + "─" * 60)
            while True:
                otra = input(
                    "¿Tienes otra actividad para registrar? (si / no): "
                ).strip().lower()
                if otra in ("si", "no"):
                    break
                print("⚠️ Opción no válida. Por favor escribe 'si' o 'no'.")

            if otra != "si":
                break

            numero += 1

        print("\n" + "═" * 60)
        print(f"  ✅  Se registraron {len(actividades)} actividad(es) en total.")
        print("═" * 60)
        return actividades

    # -----------------------------------------------------------------------
    # E.1  CAPTURAR CONTEXTO DEL COLABORADOR (conversacional, sin widgets)
    # -----------------------------------------------------------------------
    def capturar_contexto_conversacional(self):
        print("\n" + "═" * 60)
        print("  👤  INFORMACIÓN DEL COLABORADOR")
        print("═" * 60)

        while True:
            cargo = ""
            while not cargo:
                cargo = input(
                    "\n¿Cuál es tu cargo actual? "
                    "(Ej: Analista de Operaciones, Coordinador de Proyectos): "
                ).strip()
                if not cargo:
                    print("⚠️ Por favor, ingresa tu cargo.")

            vicepresidencia = ""
            while not vicepresidencia:
                vicepresidencia = input(
                    "\n¿A qué vicepresidencia o área perteneces? "
                    "(Ej: Tecnología, Operaciones, Finanzas): "
                ).strip()
                if not vicepresidencia:
                    print("⚠️ Por favor, ingresa la vicepresidencia.")

            print("\n" + "─" * 60)
            print("  Verifica que la información esté correcta:")
            print(f"    Cargo           : {cargo}")
            print(f"    Vicepresidencia : {vicepresidencia}")
            print("─" * 60)

            confirma = input(
                "\n¿La información es correcta? (si / no): "
            ).strip().lower()

            if confirma == "si":
                print(f"\n✅ Contexto registrado: {cargo} — {vicepresidencia}")
                return {"cargo": cargo, "vicepresidencia": vicepresidencia}

            print("\n🔄 Volvamos a ingresar la información.\n")

    # -----------------------------------------------------------------------
    # ★ NUEVO v5 — SELECCIÓN DE MODO AL INICIO DEL PIPELINE
    # -----------------------------------------------------------------------
    def _seleccionar_modo_inicio(self):
        """
        Pregunta al usuario si quiere:
          1) Solo registrar actividades manualmente, o
          2) Registrar actividades Y cargar la transcripción de Teams.

        Retorna
        -------
        str  "manual" | "con_transcripcion"
        """
        print("\n" + "═" * 60)
        print("  🚀  BIENVENIDO — LEVANTAMIENTO DE CARGAS LABORALES")
        print("═" * 60)
        print(
            "\n¿Cómo deseas iniciar el proceso?\n"
            "\n  1) Solo registrar mis actividades manualmente"
            "\n  2) Registrar actividades y además cargar la transcripción"
            "\n     de una reunión de Microsoft Teams (archivo .vtt)"
        )

        while True:
            opcion = input("\nSelecciona una opción (1 / 2): ").strip()
            if opcion == "1":
                print("\n✅ Modo seleccionado: Registro manual de actividades.\n")
                return "manual"
            elif opcion == "2":
                print(
                    "\n✅ Modo seleccionado: Registro manual + "
                    "transcripción de Teams.\n"
                )
                return "con_transcripcion"
            else:
                print("⚠️ Opción no válida. Por favor selecciona 1 o 2.")

    # -----------------------------------------------------------------------
    # ★ NUEVO v7 — CONSOLIDACIÓN DE ACTIVIDADES REPETIDAS EN LA TRANSCRIPCIÓN
    # -----------------------------------------------------------------------
    def _consolidar_duplicados_vtt(self, actividades_crudas):
        """
        Detecta actividades repetidas en la lista extraída del VTT,
        calcula una duración representativa para cada grupo y propone
        la consolidación al usuario antes de continuar.

        CRITERIO DE DURACIÓN para actividades agrupadas
        ------------------------------------------------
        Se usa la **mediana recortada al 20%** (trimmed median):
          1. Se ordenan las duraciones del grupo.
          2. Se descartan el 20% más bajo y el 20% más alto
             (protege contra valores atípicos por interrupciones o
             charlas que alargaron/acortaron el registro).
          3. Se toma la mediana de los valores restantes.
          4. Si el grupo tiene 1 o 2 registros no se recorta,
             se usa directamente la mediana simple.
        Esto es más robusto que el promedio cuando hay outliers
        (ej: una ejecución que duró 5 min por interrupción y otra
        que duró 40 min por una extensión excepcional).

        Flujo
        -----
        1. Envía todas las actividades al LLM para que detecte grupos
           de actividades similares/repetidas y proponga nombres limpios.
        2. Para cada grupo, calcula la duración representativa.
        3. Muestra al usuario una tabla con los grupos detectados
           (cuántas veces aparece, duraciones individuales, duración
           propuesta) y pide confirmación grupo por grupo.
        4. Si el usuario rechaza una consolidación, conserva las
           ocurrencias originales por separado.

        Retorna
        -------
        list[dict]  — lista final (puede tener menos items que la entrada
                      si se consolidaron grupos, o la misma cantidad si
                      el usuario rechazó todas las consolidaciones).
        """
        import math

        if len(actividades_crudas) <= 1:
            return actividades_crudas

        # ── Paso 1: LLM detecta grupos de actividades similares ───────────
        print("\n🔍 Analizando actividades repetidas en la transcripción...")

        prompt = f"""
Eres un analista experto en Levantamiento de Cargas Laborales.

Se te entrega una lista de actividades extraídas de una grabación de
Microsoft Teams. Algunas pueden estar repetidas (la misma tarea realizada
varias veces durante la reunión con texto ligeramente distinto).

Tu tarea:
1. Identifica grupos de actividades que sean la MISMA tarea.
   Considera como "misma tarea" cuando el texto es semánticamente
   equivalente aunque las palabras difieran levemente
   (ej: "revisión de informe" y "revisar informe" → mismo grupo).
2. Asigna a cada actividad un "grupo_id" (entero, empezando en 0).
   Actividades únicas (sin repetición) van cada una en su propio grupo.
3. Para cada grupo propón un "nombre_consolidado" limpio y descriptivo.

⚠️ REGLAS ESTRICTAS:
- Devuelve EXCLUSIVAMENTE un JSON válido, sin texto ni markdown.
- La lista de salida debe tener EXACTAMENTE el mismo número de elementos
  que la lista de entrada, en el mismo orden.

Formato de salida:
[
  {{
    "indice_original": 0,
    "grupo_id": 0,
    "nombre_consolidado": "nombre limpio del grupo"
  }},
  ...
]

Actividades de entrada (índice, texto, duración en minutos):
{json.dumps(
    [{"indice": i, "texto": a.get("actividad",""), "duracion_min": a.get("duracion_min", 0)}
     for i, a in enumerate(actividades_crudas)],
    indent=2, ensure_ascii=False
)}
"""
        try:
            reply = llm.invoke(prompt)
            clasificaciones = self._parse_json_seguro(reply.content)
        except Exception:
            # Si el LLM falla, devolver las actividades sin consolidar
            print("⚠️ No se pudo analizar duplicados. Se conservan todas las actividades.")
            return actividades_crudas

        # ── Paso 2: Agrupar por grupo_id ──────────────────────────────────
        grupos = {}   # grupo_id → {"nombre": str, "items": [dict]}
        for clf in clasificaciones:
            idx   = clf.get("indice_original", 0)
            gid   = clf.get("grupo_id", idx)
            nombre = clf.get("nombre_consolidado", "")
            if gid not in grupos:
                grupos[gid] = {"nombre": nombre, "items": []}
            if idx < len(actividades_crudas):
                grupos[gid]["items"].append(actividades_crudas[idx])

        # Separar grupos con duplicados vs únicos
        grupos_con_dup = {gid: g for gid, g in grupos.items() if len(g["items"]) > 1}
        grupos_unicos  = {gid: g for gid, g in grupos.items() if len(g["items"]) == 1}

        if not grupos_con_dup:
            print("✅ No se detectaron actividades repetidas.")
            return actividades_crudas

        # ── Paso 3: Calcular duración representativa (trimmed median) ─────
        def duracion_representativa(duraciones):
            """Mediana recortada al 20% (trimmed median)."""
            n = len(duraciones)
            if n <= 2:
                duraciones_sorted = sorted(duraciones)
                mid = n // 2
                if n % 2 == 0:
                    return round((duraciones_sorted[mid - 1] + duraciones_sorted[mid]) / 2)
                return duraciones_sorted[mid]
            corte = max(1, math.floor(n * 0.20))
            recortadas = sorted(duraciones)[corte: n - corte]
            mid = len(recortadas) // 2
            if len(recortadas) % 2 == 0:
                return round((recortadas[mid - 1] + recortadas[mid]) / 2)
            return recortadas[mid]

        # ── Paso 4: Mostrar propuestas y pedir confirmación ───────────────
        print("\n" + "═" * 60)
        print("  🔁  ACTIVIDADES REPETIDAS DETECTADAS")
        print("═" * 60)
        print(
            f"\n  Se encontraron {len(grupos_con_dup)} grupo(s) de actividades "
            "que parecen repetirse.\n"
            "  Por cada grupo se propondrá consolidarlas en UNA sola actividad\n"
            "  usando la duración más representativa del grupo.\n"
        )

        actividades_resultado = []

        for gid, grupo in sorted(grupos_con_dup.items()):
            items   = grupo["items"]
            nombre  = grupo["nombre"]
            durs    = [a.get("duracion_min", 0) for a in items]
            dur_rep = duracion_representativa(durs)

            print("─" * 60)
            print(f"  Grupo: \"{nombre}\"")
            print(f"  Apariciones: {len(items)}")
            print(f"  Duraciones individuales (min): {durs}")
            print(
                f"  Duración representativa propuesta: {dur_rep} min\n"
                f"  (mediana recortada — descarta tiempos atípicos)"
            )
            print("─" * 60)

            while True:
                resp = input(
                    "  ¿Consolidar en UNA actividad con esa duración? (si / no): "
                ).strip().lower()
                if resp in ("si", "no"):
                    break
                print("  ⚠️ Por favor escribe 'si' o 'no'.")

            if resp == "si":
                # Crear una sola actividad consolidada
                actividades_resultado.append({
                    "actividad":    nombre,
                    "inicio":       items[0].get("inicio", ""),
                    "fin":          items[-1].get("fin", ""),
                    "duracion_min": dur_rep,
                    "_consolidada": True,
                    "_ocurrencias": len(items),
                    "_duraciones_originales": durs,
                })
                print(
                    f"  ✅ Consolidada como \"{nombre}\" "
                    f"({len(items)} ocurrencias → {dur_rep} min)."
                )
            else:
                # Conservar todas las ocurrencias originales
                for act in items:
                    actividades_resultado.append(act)
                print(f"  ↩️ Se conservan las {len(items)} ocurrencias por separado.")

        # Agregar actividades únicas (sin cambios, preservando orden original)
        for gid, grupo in sorted(grupos_unicos.items()):
            actividades_resultado.extend(grupo["items"])

        # Re-ordenar por timestamp de inicio para mantener la cronología
        def _ts_to_secs(ts):
            try:
                partes = ts.split(":")
                return int(partes[0]) * 3600 + int(partes[1]) * 60 + int(partes[2])
            except Exception:
                return 0

        actividades_resultado.sort(key=lambda a: _ts_to_secs(a.get("inicio", "0:0:0")))

        print("\n" + "═" * 60)
        n_orig = len(actividades_crudas)
        n_final = len(actividades_resultado)
        print(
            f"  ✅  Resultado: {n_orig} actividades detectadas → "
            f"{n_final} actividades tras consolidación."
        )
        print("═" * 60)

        return actividades_resultado

    def _cargar_vtt_conversacional(self, contexto, procesos_registrados=None):
        """
        Solicita la ruta del archivo VTT de Teams, lo procesa, muestra un
        resumen tabular de las actividades detectadas y ofrece al usuario
        completar los campos faltantes (frecuencia, volumen, duración,
        autonomía, proceso del área, observaciones) para cada una.

        Para cada actividad completada aplica el mismo pipeline de análisis
        que las actividades manuales:
          1. Evaluación de relevancia y completitud (nombre + descripción).
          2. Validación de coherencia (volumen × duración vs. jornada).
          3. Resumen de confirmación con editor de campos si el usuario
             dice "no".

        Las actividades confirmadas quedan listas para ser enriquecidas con
        IA y añadidas al Excel final, exactamente igual que las manuales.

        Parámetros
        ----------
        contexto             : dict  — {"cargo": ..., "vicepresidencia": ...}
        procesos_registrados : list[str] | None
            Procesos del área ya capturados en otras actividades previas.

        Retorna
        -------
        list[dict]  — actividades de la transcripción con todos sus campos
                      completados por el usuario (puede ser lista vacía).
        """
        if procesos_registrados is None:
            procesos_registrados = []

        print("\n" + "─" * 60)
        print("  📂  CARGA DE TRANSCRIPCIÓN DE TEAMS (.vtt)")
        print("─" * 60)

        # ── Cargar y parsear el archivo ────────────────────────────────────
        while True:
            ruta = input(
                "\nIngresa la ruta del archivo .vtt "
                "(o presiona Enter para omitir este paso): "
            ).strip()

            if not ruta:
                print("⚠️ No se cargó ningún archivo. Continuando sin transcripción.")
                return []

            try:
                with open(ruta, "r", encoding="utf-8") as f:
                    contenido = f.read()
                actividades_crudas = self.procesar_archivo(contenido)
                break
            except FileNotFoundError:
                print(f"⚠️ Archivo no encontrado: {ruta}. Intenta de nuevo.")
            except Exception as e:
                print(f"⚠️ Error al procesar el archivo: {e}. Intenta de nuevo.")

        if not actividades_crudas:
            print("\n⚠️ No se detectaron actividades en la transcripción.")
            return []

        # ── Consolidar actividades repetidas antes de mostrar el resumen ──
        actividades_crudas = self._consolidar_duplicados_vtt(actividades_crudas)

        # ── Mostrar resumen tabular de lo extraído ─────────────────────────
        print("\n" + "═" * 60)
        print("  📋  RESUMEN DE ACTIVIDADES DETECTADAS EN LA TRANSCRIPCIÓN")
        print("═" * 60)
        print(f"  Se encontraron {len(actividades_crudas)} actividad(es):\n")

        col_n   = len(str(len(actividades_crudas)))  # ancho del índice
        col_dur = 10  # ancho columna duración
        col_ocu = 12  # ancho columna ocurrencias
        ancho_desc = 40

        encabezado = (
            f"  {'#':>{col_n}}  "
            f"{'Actividad detectada':<{ancho_desc}}  "
            f"{'Duración':>{col_dur}}  "
            f"{'Ocurrencias':>{col_ocu}}"
        )
        print(encabezado)
        print("  " + "─" * (col_n + 2 + ancho_desc + 2 + col_dur + 2 + col_ocu))

        for i, act in enumerate(actividades_crudas, start=1):
            desc = act.get("actividad", "Sin descripción")
            if len(desc) > ancho_desc:
                desc = desc[: ancho_desc - 3] + "..."
            dur      = act.get("duracion_min", 0)
            dur_txt  = f"{dur} min"
            ocu      = act.get("_ocurrencias", 1)
            ocu_txt  = f"{'★ ' + str(ocu) + 'x' if ocu > 1 else '—':>{col_ocu}}"
            print(
                f"  {i:>{col_n}}  "
                f"{desc:<{ancho_desc}}  "
                f"{dur_txt:>{col_dur}}  "
                f"{ocu_txt}"
            )

        print("  " + "─" * (col_n + 2 + ancho_desc + 2 + col_dur + 2 + col_ocu))
        print(
            "\n  Nota: 'Actividad detectada' corresponde al fragmento de texto\n"
            "  de la grabación en el que se identificó el inicio de una tarea.\n"
            "  La duración es el tiempo representativo por ejecución.\n"
            "  ★ = actividad consolidada (aparecía varias veces en la grabación)."
        )
        print("═" * 60)

        # ── Preguntar si desea completar las actividades ───────────────────
        while True:
            completar = input(
                "\n¿Deseas completar los campos faltantes de estas actividades "
                "para incluirlas en el levantamiento? (si / no): "
            ).strip().lower()
            if completar in ("si", "no"):
                break
            print("⚠️ Por favor escribe 'si' o 'no'.")

        if completar != "si":
            print(
                "\nℹ️  Las actividades de la transcripción se usarán solo como\n"
                "   referencia y NO se incluirán en el Excel."
            )
            return []

        # ── Completar campos de cada actividad detectada ───────────────────
        print("\n" + "═" * 60)
        print("  ✍️  COMPLETAR CAMPOS DE ACTIVIDADES DE LA TRANSCRIPCIÓN")
        print("═" * 60)
        print(
            "\n  Para cada actividad detectada podrás:\n"
            "  • Ajustar su nombre/descripción si el texto extraído no es claro.\n"
            "  • Ingresar los campos operativos que faltan.\n"
            "  • Descartarla si no deseas incluirla."
        )

        actividades_completadas = []

        for i, act_cruda in enumerate(actividades_crudas, start=1):
            print("\n" + "─" * 60)
            print(f"  ACTIVIDAD {i} DE {len(actividades_crudas)}")
            print("─" * 60)
            print(f"  Texto extraído : {act_cruda.get('actividad', '')}")
            print(f"  Duración       : {act_cruda.get('duracion_min', 0)} min")
            if not act_cruda.get("_consolidada"):
                print(f"  Inicio         : {act_cruda.get('inicio', '')}  "
                      f"Fin: {act_cruda.get('fin', '')}")
            else:
                print(
                    f"  ★ Consolidada de {act_cruda.get('_ocurrencias', '?')} "
                    f"ocurrencias — duraciones originales: "
                    f"{act_cruda.get('_duraciones_originales', [])} min"
                )

            # Opción de descartar antes de completar
            while True:
                incluir = input(
                    "\n  ¿Incluir esta actividad? (si / no): "
                ).strip().lower()
                if incluir in ("si", "no"):
                    break
                print("  ⚠️ Por favor escribe 'si' o 'no'.")

            if incluir != "si":
                print("  🗑️  Actividad descartada.")
                continue

            # ── El usuario completa los campos en un loop hasta confirmar ──
            # (igual que en capturar_actividades_conversacional)
            while True:

                # ── Nombre (con posibilidad de ajustarlo) ──────────────────
                nombre_sugerido = act_cruda.get("actividad", "").strip()
                print(f"\n  Nombre sugerido: \"{nombre_sugerido}\"")
                ajuste = input(
                    "  ¿Quieres ajustar el nombre? (escribe el nuevo nombre "
                    "o presiona Enter para conservarlo): "
                ).strip()
                nombre = ajuste if ajuste else nombre_sugerido

                # ── Duración (pre-cargada desde el VTT, confirmable) ───────
                dur_vtt = act_cruda.get("duracion_min", 0)
                print(f"\n  La duración detectada es {dur_vtt} min.")
                while True:
                    resp = input(
                        "  ¿Es correcta? (si) o ingresa la duración real en minutos: "
                    ).strip()
                    if resp.lower() == "si" or resp == "":
                        duracion_min = dur_vtt
                        break
                    elif resp.isdigit() and int(resp) > 0:
                        duracion_min = int(resp)
                        break
                    else:
                        print("  ⚠️ Ingresa 'si' o un número entero mayor a 0.")

                # ── Frecuencia ─────────────────────────────────────────────
                frecuencia = None
                opciones_str = "  ".join(
                    f"{j+1}) {f}" for j, f in enumerate(self.FRECUENCIAS_VALIDAS)
                )
                print(f"\n  ¿Con qué frecuencia realizas esta actividad?")
                print(f"    {opciones_str}")
                while frecuencia is None:
                    resp = input("  Selecciona el número o escribe la frecuencia: ").strip()
                    if resp.isdigit() and 1 <= int(resp) <= len(self.FRECUENCIAS_VALIDAS):
                        frecuencia = self.FRECUENCIAS_VALIDAS[int(resp) - 1]
                    elif resp.capitalize() in self.FRECUENCIAS_VALIDAS:
                        frecuencia = resp.capitalize()
                    else:
                        print(
                            f"  ⚠️ Opción no válida. Elige un número del "
                            f"1 al {len(self.FRECUENCIAS_VALIDAS)}."
                        )

                # ── Volumen ────────────────────────────────────────────────
                volumen = None
                print(
                    f"\n  ¿Cuántas veces realizas esta actividad por periodo "
                    f"({frecuencia.lower()})?"
                )
                while volumen is None:
                    resp = input("  Volumen: ").strip()
                    if resp.isdigit() and int(resp) > 0:
                        volumen = int(resp)
                    else:
                        print("  ⚠️ Ingresa un número entero mayor a 0.")

                # ── Autonomía ──────────────────────────────────────────────
                autonomia_opciones = list(range(5, 105, 5))
                autonomia = None
                fila_aut = "    " + "  ".join(f"{p}%" for p in autonomia_opciones)
                print("\n  ¿Qué porcentaje de esta actividad recae sobre ti?")
                print(f"  Opciones (de 5 en 5):\n{fila_aut}")
                while autonomia is None:
                    resp = input("  Autonomía (%): ").strip().replace("%", "").strip()
                    if resp.isdigit() and int(resp) in autonomia_opciones:
                        autonomia = int(resp)
                    else:
                        print("  ⚠️ Elige un valor de 5 en 5 entre 5% y 100%.")

                # ── Proceso del área (con reutilización) ───────────────────
                procesos_acumulados = list(procesos_registrados) + [
                    a.get("proceso_area", "")
                    for a in actividades_completadas
                    if a.get("proceso_area")
                ]
                proceso_area = self._preguntar_proceso_area(procesos_acumulados)

                # ── Observaciones ──────────────────────────────────────────
                observaciones = input(
                    "\n  Observaciones adicionales (Enter para omitir): "
                ).strip()

                # ── Construir dict temporal con todos los campos ───────────
                actividad = {
                    "nombre":        nombre,
                    "frecuencia":    frecuencia,
                    "volumen":       volumen,
                    "duracion_min":  duracion_min,
                    "autonomia":     autonomia,
                    "proceso_area":  proceso_area,
                    "observaciones": observaciones,
                    "_origen":       "transcripcion_teams",
                    # Preservar metadatos de consolidación si los había
                    **{k: act_cruda[k] for k in (
                        "_consolidada", "_ocurrencias", "_duraciones_originales"
                    ) if k in act_cruda},
                }

                # ── ANÁLISIS 1: Relevancia y completitud (nombre + descripción)
                print("\n🔍 Evaluando relevancia y completitud de la actividad...")
                evaluacion = self._evaluar_relevancia_y_detalle(
                    actividad,
                    contexto.get("cargo", ""),
                    contexto.get("vicepresidencia", "")
                )
                accion, actividad = self._manejar_alerta_relevancia(
                    actividad, evaluacion
                )
                if accion == "descartar":
                    print("  🗑️  Actividad descartada por el análisis de relevancia.")
                    actividad = None
                    break   # salir del while True interno → siguiente act_cruda

                # ── ANÁLISIS 2: Coherencia de datos (volumen × duración)
                min_acum_previo = self._calcular_minutos_diarios_acumulados(
                    actividades_completadas
                )
                min_disponibles = max(8.5 * 60 - min_acum_previo, 0)

                print("\n🔍 Revisando coherencia de los datos ingresados...")
                hay_incoherencia, explicacion, recomendaciones = (
                    self._validar_coherencia_actividad_ia(
                        actividad,
                        min_disponibles=min_disponibles if min_acum_previo > 0 else None,
                    )
                )
                if hay_incoherencia:
                    actividad = self._mostrar_incoherencia_y_corregir(
                        actividad,
                        explicacion,
                        recomendaciones,
                        min_disponibles=min_disponibles if min_acum_previo > 0 else None,
                    )
                else:
                    print("✅ Coherencia OK — los datos ingresados son consistentes.")

                # ── ANÁLISIS 3: Resumen + confirmación (con editor si dice "no")
                confirmada = self._confirmar_actividad(
                    actividad           = actividad,
                    numero              = i,
                    actividades_previas = actividades_completadas,
                    contexto            = contexto,
                )
                if confirmada:
                    break   # datos aceptados → salir del while True interno
                # Si _confirmar_actividad retorna False (flujo legacy),
                # el bucle vuelve a pedir todos los campos desde el inicio.

            # ── Agregar a la lista si no fue descartada ────────────────────
            if actividad is not None:
                actividades_completadas.append(actividad)
                print(f"  ✅ Actividad {i} completada y añadida al levantamiento.")

        print("\n" + "═" * 60)
        print(
            f"  ✅  {len(actividades_completadas)} de {len(actividades_crudas)} "
            "actividad(es) de la transcripción serán incluidas."
        )
        print("═" * 60)

        return actividades_completadas

    # -----------------------------------------------------------------------
    # G. GENERAR EXCEL DESCARGABLE CON LOS DATOS DEL COLABORADOR
    # -----------------------------------------------------------------------
    def generar_excel_actividades(
        self,
        actividades,
        contexto,
        ruta_plantilla=None,
        ruta_salida="levantamiento_cargas.xlsx"
    ):
        from openpyxl import Workbook, load_workbook
        from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
        from openpyxl.utils import get_column_letter

        vicepresidencia = contexto.get("vicepresidencia", "")

        if ruta_plantilla:
            wb = load_workbook(ruta_plantilla)
            ws = wb["FORMATO"] if "FORMATO" in wb.sheetnames else wb.active

            fila_encabezado = None
            for fila in ws.iter_rows():
                for celda in fila:
                    if (
                        celda.value
                        and "DESCRIPCI" in str(celda.value).upper()
                        and "ACTIVIDAD" in str(celda.value).upper()
                    ):
                        fila_encabezado = celda.row
                        break
                if fila_encabezado:
                    break

            if fila_encabezado is None:
                raise ValueError(
                    "No se encontró la fila de encabezados en la plantilla."
                )

            col_map = {}
            for celda in ws[fila_encabezado]:
                if celda.value:
                    key = str(celda.value).strip().upper()
                    col_map[key] = celda.column

            fila_inicio = fila_encabezado + 1

            for r in range(fila_inicio, ws.max_row + 1):
                for c in range(1, ws.max_column + 1):
                    ws.cell(row=r, column=c).value = None

            for i, act in enumerate(actividades):
                fila = fila_inicio + i
                datos = _actividad_a_fila(act, numero=i + 1,
                                          vicepresidencia=vicepresidencia)

                for nombre_col_normalizado, valor in datos.items():
                    col_idx = col_map.get(nombre_col_normalizado)
                    if col_idx:
                        celda = ws.cell(row=fila, column=col_idx, value=valor)
                        if "AUTONOM" in nombre_col_normalizado:
                            celda.number_format = "0%"
                        if nombre_col_normalizado in (
                            "NO.", "FRECUENCIA", "VOL.(SEGÚN FRECUENCIA)",
                            "TIEMPO ESTIMADO POR UNIDAD EN MINUTOS",
                            "AUTONOMÍA DE LA TAREA (0-100%)",
                            "TIPO DE ACTIVIDAD (DENTRO DEL PHVA)",
                        ):
                            celda.alignment = Alignment(
                                horizontal="center", vertical="center"
                            )
                        else:
                            celda.alignment = Alignment(
                                vertical="center", wrap_text=True
                            )

        else:
            ENCABEZADOS = [
                "NO.",
                "DESCRIPCIÓN DE LA ACTIVIDAD",
                "UNIDAD DE MEDIDA",
                "FRECUENCIA",
                "VOL.(SEGÚN FRECUENCIA)",
                "TIEMPO ESTIMADO POR UNIDAD EN MINUTOS",
                "PROCESO DEL ÁREA",
                "AUTONOMÍA DE LA TAREA (0-100%)",
                "TIPO DE ACTIVIDAD (DENTRO DEL PHVA)",
                "VICEPRESIDENCIA",
                "OBSERVACIONES",
            ]
            ANCHOS = [7, 55, 22, 18, 22, 22, 22, 18, 18, 20, 25]

            borde = Border(
                top    = Side(style="thin"),
                bottom = Side(style="thin"),
                left   = Side(style="thin"),
                right  = Side(style="thin"),
            )

            wb = Workbook()
            ws = wb.active
            ws.title = "FORMATO"

            ws.append(ENCABEZADOS)
            for col_idx, _ in enumerate(ENCABEZADOS, start=1):
                c = ws.cell(row=1, column=col_idx)
                c.font      = Font(bold=True, size=14)
                c.alignment = Alignment(
                    horizontal="center", vertical="center", wrap_text=True
                )
                c.border    = borde
            ws.row_dimensions[1].height = 59

            for col_idx, ancho in enumerate(ANCHOS, start=1):
                ws.column_dimensions[get_column_letter(col_idx)].width = ancho

            for i, act in enumerate(actividades):
                datos = _actividad_a_fila(act, numero=i + 1,
                                          vicepresidencia=vicepresidencia)
                fila_vals = [datos.get(enc.upper(), "") for enc in ENCABEZADOS]
                ws.append(fila_vals)
                fila_actual = ws.max_row
                for col_idx in range(1, len(ENCABEZADOS) + 1):
                    c = ws.cell(row=fila_actual, column=col_idx)
                    c.border    = borde
                    c.alignment = Alignment(vertical="center", wrap_text=True)
                    if col_idx == 8:
                        c.number_format = "0%"

        wb.save(ruta_salida)
        print(f"\n📥 Excel generado: {ruta_salida}")
        return ruta_salida

    # -----------------------------------------------------------------------
    # H. GUARDAR EN BIGQUERY
    # -----------------------------------------------------------------------
    def guardar_en_bigquery(
        self,
        actividades,
        contexto,
        project_id,
        dataset_id,
        table_id,
        credentials_path=None
    ):
        from google.cloud import bigquery
        from google.oauth2 import service_account
        from datetime import datetime, timezone

        if credentials_path:
            creds  = service_account.Credentials.from_service_account_file(
                credentials_path,
                scopes=["https://www.googleapis.com/auth/cloud-platform"]
            )
            client = bigquery.Client(project=project_id, credentials=creds)
        else:
            client = bigquery.Client(project=project_id)

        tabla_ref   = f"{project_id}.{dataset_id}.{table_id}"
        timestamp   = datetime.now(timezone.utc).isoformat()

        filas = []
        for act in actividades:
            filas.append({
                "timestamp_carga":       timestamp,
                "cargo":                 contexto.get("cargo", ""),
                "vicepresidencia":       contexto.get("vicepresidencia", ""),
                "descripcion_actividad": act.get("nombre", ""),
                "frecuencia":            act.get("frecuencia", ""),
                "volumen":               act.get("volumen"),
                "duracion_min":          act.get("duracion_min"),
                "personas":              act.get("personas"),
                "autonomia_pct":         act.get("autonomia"),
                "proceso_area":          act.get("proceso_area", ""),
                "unidad_medida":         act.get("unidad_medida", ""),
                "phva":                  act.get("phva", ""),
                "observaciones":         act.get("observaciones", ""),
            })

        errores = client.insert_rows_json(tabla_ref, filas)

        if not errores:
            print(
                f"\n✅ {len(filas)} fila(s) insertada(s) en "
                f"{tabla_ref}"
            )
        else:
            print(f"\n⚠️ Errores al insertar en BigQuery:")
            for err in errores:
                print(f"   {err}")

    # -----------------------------------------------------------------------
    # F. PIPELINE COMPLETO PARA ACTIVIDADES CONVERSACIONALES
    #    MODIFICADO v5: incluye selección de modo al inicio
    # -----------------------------------------------------------------------
    def procesar_actividades_conversacional(
        self,
        ruta_plantilla_excel=None,
        ruta_salida_excel="levantamiento_cargas.xlsx",
        bq_project_id=None,
        bq_dataset_id=None,
        bq_table_id="levantamiento_cargas",
        bq_credentials_path=None,
    ):
        """
        Pipeline de extremo a extremo para el flujo conversacional.

        Flujo v6:
        0. Pregunta al usuario si quiere solo registrar actividades o
           también cargar una transcripción de Teams.
        1. Solicita cargo y vicepresidencia (conversacional).
        2. (Opcional) Carga y procesa el archivo VTT de Teams:
           - Muestra resumen tabular de actividades detectadas.
           - Permite completar los campos faltantes de cada una.
           - Las actividades completadas se unen a las manuales.
        3. Captura actividades manuales con:
           - Reutilización de procesos del área ya registrados
             (incluye los de las actividades VTT completadas).
           - Detección de actividades irrelevantes o sin suficiente
             detalle evaluando nombre + descripción en conjunto.
           - Validación de coherencia y análisis interno de carga.
           - Editor de campos individuales cuando el usuario dice "no"
             en la confirmación por actividad.
           - Pregunta de "otra actividad" que solo acepta si/no.
        4. Consolida actividades VTT completadas + manuales.
        5. Enriquece con IA (unidad_medida + phva).
        6. Fusiona datos base + IA.
        7. Muestra resumen global y pide confirmación final.
        8. Genera Excel descargable (ambas fuentes incluidas).
        9. Inserta en BigQuery (si se configuran los parámetros BQ).
        """
        # PASO 0 — Selección de modo
        modo = self._seleccionar_modo_inicio()

        # PASO 1 — Capturar contexto del colaborador
        contexto = self.capturar_contexto_conversacional()

        # PASO 2 (opcional) — Cargar transcripción de Teams
        actividades_vtt_completadas = []
        if modo == "con_transcripcion":
            # En este punto aún no hay actividades manuales, lista vacía
            actividades_vtt_completadas = self._cargar_vtt_conversacional(
                contexto             = contexto,
                procesos_registrados = [],
            )
            if actividades_vtt_completadas:
                print(
                    f"\nℹ️  {len(actividades_vtt_completadas)} actividad(es) de la "
                    "transcripción se unirán a las que registres manualmente."
                )

        # PASO 3 — Captura conversacional con análisis interno por actividad.
        # Los procesos del área de las actividades VTT ya completadas se
        # pasan para que estén disponibles en la selección de proceso.
        procesos_de_vtt = [
            a.get("proceso_area", "")
            for a in actividades_vtt_completadas
            if a.get("proceso_area")
        ]

        # Inyectamos los procesos VTT en el estado inicial de la captura
        # conversacional pasándolos como contexto adicional.
        actividades_base = self._capturar_actividades_conversacional_con_procesos(
            contexto             = contexto,
            procesos_iniciales   = procesos_de_vtt,
        )

        # PASO 4 — Consolidar: VTT completadas + manuales
        todas_las_actividades_base = actividades_vtt_completadas + actividades_base

        if not todas_las_actividades_base:
            print("\n⚠️ No se registraron actividades.")
            return []

        # PASO 5 — Enriquecimiento IA sobre el conjunto completo
        print("\n🤖 Enriqueciendo actividades con IA...")
        actividades_ia = self.enriquecer_actividades(
            todas_las_actividades_base,
            contexto["cargo"],
            contexto["vicepresidencia"],
        )

        # PASO 6 — Fusión
        actividades_finales = self.fusionar_inputs_usuario_nodiarias(
            todas_las_actividades_base, actividades_ia
        )

        # PASO 7 — Resumen global y confirmación final
        while True:
            print("\n📋 Generando resumen para tu confirmación final...")
            resumen = self.resumen_para_confirmacion(actividades_finales, contexto)
            confirmado = self.confirmar_informacion(resumen)

            if confirmado:
                break

            print("\n✏️ Abriendo editor de actividades...")
            actividades_finales = self.editar_actividades(actividades_finales)

            print(
                "\n🤖 Re-analizando actividades con la información actualizada..."
            )
            actividades_ia_nuevo = self.enriquecer_actividades(
                actividades_finales,
                contexto["cargo"],
                contexto["vicepresidencia"],
            )
            for act, ia in zip(actividades_finales, actividades_ia_nuevo):
                act["unidad_medida"] = ia.get("unidad_medida", act.get("unidad_medida", ""))
                act["phva"]          = ia.get("phva",          act.get("phva", ""))

            print("\n✅ Análisis actualizado. Generando nuevo resumen...")

        # PASO 8 — Generar Excel (todas las fuentes incluidas)
        self.generar_excel_actividades(
            actividades    = actividades_finales,
            contexto       = contexto,
            ruta_plantilla = ruta_plantilla_excel,
            ruta_salida    = ruta_salida_excel,
        )

        # PASO 9 — BigQuery (solo si se configuraron parámetros)
        if bq_project_id and bq_dataset_id:
            self.guardar_en_bigquery(
                actividades      = actividades_finales,
                contexto         = contexto,
                project_id       = bq_project_id,
                dataset_id       = bq_dataset_id,
                table_id         = bq_table_id,
                credentials_path = bq_credentials_path,
            )

        return actividades_finales


# ---------------------------------------------------------------------------
# Función auxiliar (module-level) para mapear un dict de actividad
# a los nombres de columna del Excel estándar
# ---------------------------------------------------------------------------
def _actividad_a_fila(act, numero=None, vicepresidencia=""):
    autonomia_raw = act.get("autonomia")
    autonomia_decimal = (autonomia_raw / 100) if autonomia_raw is not None else ""

    return {
        "NO.":                                   numero if numero is not None else "",
        "DESCRIPCIÓN DE LA ACTIVIDAD":           act.get("nombre", ""),
        "UNIDAD DE MEDIDA":                      act.get("unidad_medida", ""),
        "FRECUENCIA":                            act.get("frecuencia", ""),
        "VOL.(SEGÚN FRECUENCIA)":                act.get("volumen", ""),
        "TIEMPO ESTIMADO POR UNIDAD EN MINUTOS": act.get("duracion_min", ""),
        "PROCESO DEL ÁREA":                      act.get("proceso_area", ""),
        "AUTONOMÍA DE LA TAREA (0-100%)":        autonomia_decimal,
        "TIPO DE ACTIVIDAD (DENTRO DEL PHVA)":   act.get("phva", ""),
        "VICEPRESIDENCIA":                       vicepresidencia,
        "OBSERVACIONES":                         act.get("observaciones", ""),
    }

#### Parte del Usuario

In [21]:
procesador = ProcesadorTranscripcionTeams()

# En vez del flujo con Excel:
actividades = procesador.procesar_actividades_conversacional()

# Ya están listas para calcular métricas:
metricas = procesador.calcular_metricas(actividades)


════════════════════════════════════════════════════════════
  🚀  BIENVENIDO — LEVANTAMIENTO DE CARGAS LABORALES
════════════════════════════════════════════════════════════

¿Cómo deseas iniciar el proceso?

  1) Solo registrar mis actividades manualmente
  2) Registrar actividades y además cargar la transcripción
     de una reunión de Microsoft Teams (archivo .vtt)



Selecciona una opción (1 / 2):  2



✅ Modo seleccionado: Registro manual + transcripción de Teams.


════════════════════════════════════════════════════════════
  👤  INFORMACIÓN DEL COLABORADOR
════════════════════════════════════════════════════════════



¿Cuál es tu cargo actual? (Ej: Analista de Operaciones, Coordinador de Proyectos):  gerente

¿A qué vicepresidencia o área perteneces? (Ej: Tecnología, Operaciones, Finanzas):  talento y administrativa



────────────────────────────────────────────────────────────
  Verifica que la información esté correcta:
    Cargo           : gerente
    Vicepresidencia : talento y administrativa
────────────────────────────────────────────────────────────


...


✅ Cambios aplicados. Revisando el resumen actualizado...

════════════════════════════════════════════════════════════
  RESUMEN — Actividad 1
════════════════════════════════════════════════════════════
  Nombre            : Análisis y envío de reporte del plan de trabajo
  Frecuencia        : Diario
  Volumen           : 2 vez/veces por periodo
  Duración          : 10 min por ejecución
  Autonomía         : 100%
  Proceso del área  : Desarrollo institucional de nuevas formas de trabajo y agilidad
════════════════════════════════════════════════════════════



¿Esta información es correcta? (si / no):  si


  ✅ Actividad 1 completada y añadida al levantamiento.

════════════════════════════════════════════════════════════
  ✅  1 de 1 actividad(es) de la transcripción serán incluidas.
════════════════════════════════════════════════════════════

ℹ️  1 actividad(es) de la transcripción se unirán a las que registres manualmente.

════════════════════════════════════════════════════════════
  🗂️  REGISTRO DE ACTIVIDADES — MODO CONVERSACIONAL
════════════════════════════════════════════════════════════

A continuación te haré preguntas sobre cada actividad que realizas.
Puedes registrar todas las que quieras.
Cuando termines, escribe 'no' cuando te pregunte si hay otra actividad.

────────────────────────────────────────────────────────────
  📋  ACTIVIDAD 1
────────────────────────────────────────────────────────────


KeyboardInterrupt: Interrupted by user


¿Cuál es la actividad 1? (describe brevemente qué haces):  sdfgs


In [22]:
actividades

[{'nombre': 'Revision de plan de trabajo individual  y envio correo',
  'duracion_min': 90,
  'unidad_medida': 'correos',
  'phva': 'Verificar',
  'frecuencia': 'Diario',
  'volumen': 2,
  'proceso_area': 'Desarrollo institucional de nuevas formas de trabajo y agilidad',
  'autonomia': 100,
  'observaciones': 'Revision de pendientes y nuevas solicitudes',
  'metricas': {'minutos_diarios': 180,
   'minutos_mes': 3780,
   'carga_w_sin_tm': 0.2864,
   'carga_trabajo_individual': 0.2864}},
 {'nombre': 'Liderar el diseño y actualización del roadmap',
  'duracion_min': 60,
  'unidad_medida': 'reuniones',
  'phva': 'Planear',
  'frecuencia': 'Semanal',
  'volumen': 1,
  'proceso_area': 'Consolidación del equipo de Cultura y Transformación',
  'autonomia': 100,
  'observaciones': 'Sesion de alineación de prioridades con el equipo de leads de agilidad',
  'metricas': {'minutos_diarios': 12.0,
   'minutos_mes': 252.0,
   'carga_w_sin_tm': 0.0191,
   'carga_trabajo_individual': 0.0191}},
 {'nombr

In [23]:
metricas

{'carga_trabajo_phva': {'Verificar': 0.2864, 'Planear': 0.0382},
 'cantidad_actividades_phva': {'Verificar': 1, 'Planear': 2},
 'porcentaje_actividades_phva': {'Verificar': 0.8824, 'Planear': 0.1176},
 'carga_trabajo_frecuencia': {'Diario': 0.2864, 'Semanal': 0.0382},
 'cantidad_actividades_frecuencia': {'Diario': 1, 'Semanal': 2},
 'porcentaje_actividades_frecuencia': {'Diario': 0.8824, 'Semanal': 0.1176},
 'carga_trabajo_proceso_area': {'Desarrollo institucional de nuevas formas de trabajo y agilidad': 0.2864,
  'Consolidación del equipo de Cultura y Transformación': 0.0382},
 'cantidad_actividades_proceso_area': {'Desarrollo institucional de nuevas formas de trabajo y agilidad': 1,
  'Consolidación del equipo de Cultura y Transformación': 2},
 'porcentaje_actividades_proceso_area': {'Desarrollo institucional de nuevas formas de trabajo y agilidad': 0.8824,
  'Consolidación del equipo de Cultura y Transformación': 0.1176},
 'total_carga_w_sin_tm': 0.3245,
 'total_carga_trabajo_indivi

#### Parte del Analista

In [179]:
resumen = procesador.resumen_para_confirmacion(
    actividades_finales_nodiarias,
    contexto
)

In [180]:
metricas = procesador.calcular_metricas(actividades_finales_final)

In [181]:
analisis = procesador.analisis_analista_ia(
        actividades_finales_final,
        metricas,
        contexto
    )

In [42]:
display(Markdown("## 📋 Actividades"))

df_act = pd.DataFrame(actividades_finales_final)
display(df_act)

display(Markdown("## 📊 Métricas por PHVA"))
df_phva = pd.DataFrame({
    "Carga de Trabajo": metricas["carga_trabajo_phva"],
    "# Act.": metricas["cantidad_actividades_phva"],
    "% Act.": metricas["porcentaje_actividades_phva"]
})
display(df_phva)

display(Markdown("## 📊 Métricas por FRECUENCIA"))
df_frecuencia = pd.DataFrame({
    "Carga de Trabajo": metricas["carga_trabajo_frecuencia"],
    "# Act.": metricas["cantidad_actividades_frecuencia"],
    "% Act.": metricas["porcentaje_actividades_frecuencia"]
})
display(df_frecuencia)

display(Markdown("## 📊 Métricas por PROCESO"))
df_proceso = pd.DataFrame({
    "Carga de Trabajo": metricas["carga_trabajo_proceso_area"],
    "# Act.": metricas["cantidad_actividades_proceso_area"],
    "% Act.": metricas["porcentaje_actividades_proceso_area"]
})
display(df_proceso)

display(Markdown("## 📊 TOTALES"))
df_totales = pd.DataFrame([{
    "Total Carga W sin TM": metricas["total_carga_w_sin_tm"],
    "Total Carga Trabajo Individual": metricas["total_carga_trabajo_individual"],
    "Minutos diarios empleados": metricas["minutos_diarios_empleados"],
    "Horas diarias requeridas": metricas["horas_diarias_requeridas"]
}])
display(df_totales)

display(Markdown("## 📊 DOTACIÓN"))
df_dotacion = pd.DataFrame([{
    "Jornada Laboral Diaria": metricas["jornada_laboral_diaria"],
    "Factor Tiempo Neto Productivo": metricas["factor_tiempo_neto_productivo"],
    "Horas Netas Efectivas por Persona": metricas["horas_netas_efectivas_por_persona"],
    "Número Personas Requeridas": metricas["numero_personas_requeridas"]
}])
display(df_dotacion)

display(Markdown("## 📊 AJUSTE FINAL"))
df_ajuste = pd.DataFrame([{
    "Tiempo Muerto": metricas["tiempo_muerto"],
    "Horas Diarias Requeridas Final": metricas["horas_diarias_requeridas_final"]
}])
display(df_ajuste)

display(Markdown("## 🧠 Análisis del Analista IA"))
display(Markdown(analisis))

## 📋 Actividades

,nombre,duracion_min,unidad_medida,phva,frecuencia,volumen,proceso_area,autonomia,observaciones,metricas
0,Revisión de plan de trabajo individual y envío...,30,informes,Verificar,Diario,3,Desarrollo institucional de nuevas formas de t...,100.0,Revision de pendientes y nuevas solicitudes.,"{'minutos_diarios': 90, 'minutos_mes': 1890, '..."
1,Liderar el diseño y actualización del roadmap...,60,reuniones,Planear,Semanal,2,Consolidación del equipo de Cultura y Transfor...,100.0,Sesion de alineación de prioridades con el equ...,"{'minutos_diarios': 24.0, 'minutos_mes': 504.0..."
2,Liderar el diseño y actualización del roadmap...,120,reuniones,Planear,Semanal,1,Consolidación del equipo de Cultura y Transfor...,65.0,Alineación Lideres Dirección Cultura y Transfo...,"{'minutos_diarios': 24.0, 'minutos_mes': 504.0..."
3,Revisión del roadmap de delivery con el equipo...,60,reuniones,Verificar,Quincenal,1,Desarrollo institucional de nuevas formas de t...,30.0,Revision de avances en plan de trabajo de agil...,"{'minutos_diarios': 6.0, 'minutos_mes': 126.0,..."
4,Revisión del roadmap de delivery con el equipo...,60,reuniones,Verificar,Quincenal,1,Desarrollo institucional de nuevas formas de t...,30.0,Revision de avances en plan de trabajo de agil...,"{'minutos_diarios': 6.0, 'minutos_mes': 126.0,..."
5,Revisión del roadmap de delivery con el equipo...,60,reuniones,Verificar,Quincenal,1,Desarrollo institucional de nuevas formas de t...,30.0,Revision de avances en plan de trabajo de agil...,"{'minutos_diarios': 6.0, 'minutos_mes': 126.0,..."
6,Revisión del roadmap de delivery con el equipo...,60,reuniones,Verificar,Quincenal,1,Desarrollo institucional de nuevas formas de t...,30.0,Revision de avances en plan de trabajo de agil...,"{'minutos_diarios': 6.0, 'minutos_mes': 126.0,..."
7,Sesion de alineación de agilidad con aval (adl),60,reuniones,Planear,Quincenal,1,Desarrollo institucional de nuevas formas de t...,20.0,Alineación de formas de trabajo con areas de a...,"{'minutos_diarios': 6.0, 'minutos_mes': 126.0,..."
8,Agile GO Frente personas,90,seguimientos,Hacer,Semanal,1,Desarrollo institucional de nuevas formas de t...,10.0,Seguimiento a la ejecución de la estrategia co...,"{'minutos_diarios': 18.0, 'minutos_mes': 378.0..."
9,Agile GO Frente empresas,90,seguimientos,Hacer,Semanal,1,Desarrollo institucional de nuevas formas de t...,10.0,Seguimiento a la ejecución de la estrategia c...,"{'minutos_diarios': 18.0, 'minutos_mes': 378.0..."


## 📊 Métricas por PHVA

,Carga de Trabajo,# Act.,% Act.
Verificar,0.1874,6,0.3252
Planear,0.1744,7,0.3026
Hacer,0.2100,4,0.3644
Actuar,0.0045,1,0.0079


## 📊 Métricas por FRECUENCIA

...

,Jornada Laboral Diaria,Factor Tiempo Neto Productivo,Horas Netas Efectivas por Persona,Número Personas Requeridas
0,8.5,0.8676,7.37,0.927


## 📊 AJUSTE FINAL

,Tiempo Muerto,Horas Diarias Requeridas Final
0,0.1324,6.84


## 🧠 Análisis del Analista IA

1. **Nivel de carga laboral (con justificación)**

La carga laboral total calculada para el cargo de Gerente en la Vicepresidencia de Talento y Administrativa es de aproximadamente **6.84 horas diarias requeridas finales**, sobre una jornada laboral estándar de **8.5 horas diarias**. Esto representa un nivel de ocupación del **80.5%** de la jornada laboral disponible (6.84/8.5). La dotación requerida es de **0.927 personas**, lo que indica que la carga está dimensionada para una sola persona, sin necesidad de refuerzos adicionales.

Este nivel de carga puede clasificarse como **ADECUADO**. La carga no es excesiva ni baja; permite un margen para imprevistos, pausas y actividades no planificadas, considerando además un factor de tiempo neto productivo del **86.76%**. La carga diaria en minutos es de **362.29 minutos** (6.04 horas netas antes de ajustar tiempos muertos), lo que es razonable para un cargo gerencial con múltiples responsabilidades estratégicas y operativas.

2. **Validación de coherencia de las métricas**

Las métricas presentadas son coherentes con un cargo gerencial de alta responsabilidad y autonomía. La autonomía promedio declarada es alta, con valores que oscilan entre el **10% y 100%**, predominando actividades con autonomía superior al 50%, lo cual es consistente con la naturaleza del cargo que implica toma de decisiones y liderazgo.

No se observan valores atípicos en duración o volumen que distorsionen la carga total. La distribución de frecuencias muestra predominancia de actividades semanales (62.93% de las actividades y 36.27% de la carga), lo cual es típico en roles gerenciales que requieren seguimiento constante pero no microgestión diaria excesiva. La carga diaria (14.32%) corresponde a una sola actividad de revisión de plan de trabajo, lo que es razonable.

Se detecta una concentración significativa de carga en el proceso "Desarrollo institucional de nuevas formas de trabajo y agilidad" (61.91% de las actividades y 35.68% de la carga), lo que es coherente con la función estratégica del cargo, aunque se debe vigilar que no genere dependencia excesiva en un solo ámbito.

3. **Interpretación del balance PHVA**

La distribución de carga por PHVA es:

- Hacer: 21.0%
- Verificar: 18.74%
- Planear: 17.44%
- Actuar: 0.45%

El rol presenta un balance relativamente equilibrado entre las fases de **Hacer**, **Verificar** y **Planear**, con una ligera preponderancia en la ejecución (Hacer) y verificación. Esto indica que el gerente no solo diseña y planifica estrategias, sino que también participa activamente en el seguimiento y control de las iniciativas.

La fase de **Actuar** está subrepresentada (0.45%), lo que puede reflejar que las acciones correctivas o de mejora continua se delegan o se integran dentro de otras fases. No se observa una sobregestión ni una baja ejecución, pero la baja carga en Actuar podría ser un área a fortalecer para asegurar la mejora continua.

4. **Riesgos operativos identificados**

- **Sobrecarga puntual en actividades de alta frecuencia**: La actividad diaria de revisión de planes y correos consume un 14.32% de la carga, lo que puede generar fatiga si se suma a otras responsabilidades urgentes.
- **Dependencia en el proceso de agilidad**: Más del 60% de las actividades están concentradas en "Desarrollo institucional de nuevas formas de trabajo y agilidad", lo que puede generar cuellos de botella si el gerente no puede atender todas las demandas.
- **Baja autonomía en algunas actividades críticas**: Actividades como "Agile GO Frente personas" y "Sesión de alineación con aval (adl)" tienen autonomías bajas (10% y 20%), lo que puede generar dependencia de otros actores y retrasos.
- **Frecuencias quincenales y trimestrales con baja carga, pero posibles picos de trabajo**: Actividades como steerco y feria Bdb, aunque poco frecuentes, requieren dedicación concentrada que puede afectar la planificación semanal.

5. **Oportunidades de automatización o mejora**

- La actividad diaria de revisión de planes y envío de correos (90 minutos diarios) es repetitiva y con alta frecuencia; podría beneficiarse de herramientas que automaticen alertas o resúmenes para optimizar tiempo.
- Actividades con baja autonomía y alta duración, como las sesiones de alineación y seguimiento (60-90 minutos), podrían mejorar su eficiencia mediante agendas más estructuradas o predefinidas.
- La gestión de iniciativas estratégicas transversales (5 casos semanales, 60 minutos diarios) representa una carga significativa (9.55%) y es 100% autónoma; sin embargo, puede ser un área para estandarizar procesos o delegar parcialmente.
- La formación continua semanal (120 minutos) es necesaria, pero podría optimizarse con formatos flexibles o integrados en la rutina.

6. **Recomendaciones finales**

- Mantener el nivel de carga actual, dado que es adecuado, pero vigilar la concentración en el proceso de agilidad para evitar dependencia excesiva y posibles cuellos de botella.
- Fomentar un mayor énfasis en la fase de Actuar para fortalecer la mejora continua y evitar que las acciones correctivas se diluyan o deleguen sin seguimiento.
- Evaluar la redistribución o delegación de actividades con baja autonomía y alta duración para mejorar la eficiencia operativa y reducir riesgos de dependencia.
- Promover la implementación de mecanismos de automatización o estandarización en actividades de alta frecuencia y repetitividad, especialmente en la revisión diaria de planes y gestión de iniciativas.
- Considerar la flexibilización o integración de la formación continua para optimizar el tiempo sin afectar la actualización profesional.
- Finalmente, asegurar que el gerente cuente con espacios para actividades no planificadas o estratégicas que no están reflejadas en la carga actual, para mantener la capacidad de respuesta y liderazgo efectivo.

In [43]:
dfs = {
    "Actividades": df_act,
    "PHVA": df_phva,
    "Dotacion": df_dotacion,
    "Ajuste": df_ajuste,
    "Frecuencia": df_frecuencia,
    "Totales": df_totales,
    "Proceso_Area": df_proceso
}

In [44]:
ruta_excel = f"analisis_carga_laboral_{contexto['cargo']}.xlsx"

with pd.ExcelWriter(ruta_excel, engine="openpyxl") as writer:
    df_act.to_excel(writer, sheet_name="Actividades", index=True)
    df_phva.to_excel(writer, sheet_name="Métricas PHVA", index=True)
    df_frecuencia.to_excel(writer, sheet_name="Frecuencias", index=True)
    df_proceso.to_excel(writer, sheet_name="Procesos", index=True)
    df_dotacion.to_excel(writer, sheet_name="Dotacion", index=False)
    df_ajuste.to_excel(writer, sheet_name="Ajuste", index=False)
    df_totales.to_excel(writer, sheet_name="Totales", index=False)
    
    lineas_analisis = analisis.split("\n")

    df_analisis = pd.DataFrame({
        "Análisis del Analista IA": lineas_analisis
    })

    df_analisis.to_excel(writer, sheet_name='Analisis IA', index=False)

In [45]:
wb = load_workbook(ruta_excel)

for sheet in wb.sheetnames:
    ws = wb[sheet]
    for col in ws.columns:
        ws.column_dimensions[col[0].column_letter].width = 25

wb.save(ruta_excel)

In [46]:
display(HTML(f"""
<a href="{ruta_excel}" download>
📥 Descargar archivo Excel de análisis de carga laboral
</a>
"""))

# Guardado en BigQuery